# What this notebook is doing (precipitation summary stats)

This notebook computes summary precipitation metrics across the datasets I have been using for Skagit.
The goal here is not event-scale comparison, but a broader summary of how the datasets compare over the longer record.

It pulls daily precipitation for each dataset and then calculates region-based annual and summary metrics for Upper Skagit, Sauk, Lower Skagit, and Full Skagit.
That includes things like average annual total precipitation and the other derived precipitation metrics saved in the output tables.

At the end it writes the annual metrics CSV, the summary metrics CSV, and region-specific summary tables so the results can be reused outside the notebook.


In [ ]:
# ============================================================
# NOTEBOOK: Skagit precipitation summary metrics across datasets
#
# Computes, for each dataset x region:
#   1) Annual Average Total Precipitation (in)
#   2) Average Annual Number of Days with >1" Total Precipitation
#   3) Average Percent Change in 1h / 6h / 24h precipitation event
#
# DEFINITION USED HERE FOR PERCENT CHANGE:
#   % change = 100 * (compare_mean - baseline_mean) / baseline_mean
#   where compare_mean and baseline_mean are based on the
#   mean annual MAXIMUM event magnitude for that duration.
#
# YEAR CONVENTION (matches your seasonal notebook):
#   meteorological-year label Y = Dec(Y-1) + Jan..Nov(Y)
#
# IMPORTANT:
# - Daily-only datasets -> 1h, 6h are NaN
# - HRRR F06 -> 6h and 24h work, 1h is NaN
# - PNNL hourly -> 1h, 6h, 24h should work
# - SNOTEL depends on station CSV cadence; this notebook treats it as daily
# - PNNL Full Skagit may be unavailable if fallback mask is used and only
#   contains 3 sub-basins
# ============================================================


import re
import sys
import time
import math
import warnings
from glob import glob
from pathlib import Path
from urllib.parse import urlparse, urlunparse

import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- Config ----------------
BASE = Path("/data0/balaji24/data")
BOUNDARY_GEO = Path("../data/GIS/SkagitBoundary.json")
HUC8_GEO     = Path("../data/GIS/SkagitSubBasin_HUC8.geojson")

P_PRISM_DIR  = BASE / "prism_ppt"
P_HRRR_DIR   = BASE / "weather_data_06"
PNNL_HIST    = BASE / "PNNL" / "historical"

UCLA_PREC_DIR   = Path("/data0/balaji24/data/ucla_era5_d02_daily/prec")
UCLA_COORD_FILE = Path("/data0/balaji24/data/ucla_era5_d02_daily/static/wrfinput_d02_coord.nc")

# optional fallback only if PNNL regionmask directly on native grid fails
PNNL_REGION_MASK = Path("/home/balaji24/skagit-met/analysis/skagit_huc8_3mask_pnnl_450x450.nc")

OUT = BASE / "derived" / "precip_summary_metrics"
OUT.mkdir(parents=True, exist_ok=True)

REGION_NAMES = ["Upper Skagit", "Sauk", "Lower Skagit", "Full Skagit"]

YEAR_MIN = 1983
YEAR_MAX = 2024

GLOBAL_START = f"{YEAR_MIN-1}-12-01"
GLOBAL_END   = f"{YEAR_MAX}-11-30"

# ----- Choose the 2 periods for percent-change metrics -----
BASELINE_YEARS = list(range(1985, 2000))   # 1985..1999
COMPARE_YEARS  = list(range(2010, 2025))   # 2010..2024

# Coverage rules
MIN_DAILY_DAYS_PER_MET_YEAR = 330
MIN_NATIVE_COVERAGE_FRAC    = 0.90

# Turn datasets on/off
RUN_DATASETS = {
    "PRISM_4KM": True,
    "DAYMET_PC": True,
    "HRRR_F06": True,
    "UCLA_ERA5_D02": True,
    "PNNL_HIST": True,
    "CONUS404": True,
    "SNOTEL": True,
}

# output files
ANNUAL_CSV  = OUT / f"precip_annual_metrics_{YEAR_MIN}_{YEAR_MAX}.csv"
SUMMARY_CSV = OUT / f"precip_summary_metrics_{YEAR_MIN}_{YEAR_MAX}.csv"


def safe_open_zarr(p: Path):
    try:
        return xr.open_zarr(p, consolidated=True)
    except Exception:
        return xr.open_zarr(p, consolidated=False)

def ensure_time_sorted_unique(da):
    if "time" not in da.dims:
        return da
    da = da.sortby("time")
    try:
        t = pd.to_datetime(da["time"].values)
        _, idx = np.unique(t, return_index=True)
        da = da.isel(time=np.sort(idx))
    except Exception:
        pass
    return da

def find_lat_lon_names(ds):
    lat_name = next((n for n in ["lat","latitude","XLAT","XLAT_M","lat2d"] if n in ds), None)
    lon_name = next((n for n in ["lon","longitude","XLONG","XLONG_M","lon2d"] if n in ds), None)
    return lat_name, lon_name

def _to_2d(arr):
    if "time" in arr.dims:
        arr = arr.isel(time=0)
    for d in list(arr.dims):
        if arr.sizes[d] == 1:
            arr = arr.isel({d: 0})
    while arr.ndim > 2:
        extra = [d for d in arr.dims if d not in ("x","y","lon","lat")]
        if not extra:
            break
        arr = arr.isel({extra[0]: 0})
    return arr

def da_to_region_df(da, dataset_name):
    """
    DataArray dims expected: time, region
    Returns tidy df: dataset, region, time, mm
    """
    da = ensure_time_sorted_unique(da)
    df = da.to_dataframe(name="mm").reset_index()
    need = ["time", "region", "mm"]
    df = df[[c for c in need if c in df.columns]].copy()
    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"])
    df["dataset"] = dataset_name
    df = df[["dataset", "region", "time", "mm"]].sort_values(["region", "time"]).reset_index(drop=True)
    return df

def series_to_time_df(s: pd.Series, dataset_name: str, region_name: str) -> pd.DataFrame:
    """
    Convert a time-indexed Series -> DataFrame with columns:
    dataset, region, time, mm
    Works even if the index name is 'datetime', 'index', or anything else.
    """
    if s is None or len(s) == 0:
        return pd.DataFrame(columns=["dataset", "region", "time", "mm"])

    s = s.sort_index()
    df = s.rename("mm").to_frame().reset_index()
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "time"})
    df["time"] = pd.to_datetime(df["time"])
    df["region"] = region_name
    df["dataset"] = dataset_name
    return df[["dataset", "region", "time", "mm"]]

def load_regions_gdf():
    """
    3 clipped HUC8 regions + Full Skagit boundary
    """
    import geopandas as gpd

    skagit_gdf = gpd.read_file(BOUNDARY_GEO).to_crs("EPSG:4326")
    huc8 = gpd.read_file(HUC8_GEO).to_crs("EPSG:4326")

    sub3 = ["Upper Skagit", "Sauk", "Lower Skagit"]
    huc8_3 = huc8[huc8["Name"].isin(sub3)].copy()
    if huc8_3.empty:
        raise ValueError(f"No matching HUC8 regions found for {sub3}. Check 'Name' field.")

    huc8_3 = gpd.overlay(huc8_3, skagit_gdf, how="intersection")
    regions_3 = huc8_3[["Name", "geometry"]].reset_index(drop=True)

    full_row = gpd.GeoDataFrame(
        {"Name": ["Full Skagit"], "geometry": [skagit_gdf.geometry.iloc[0]]},
        crs="EPSG:4326",
    )

    regions_gdf = pd.concat([regions_3, full_row], ignore_index=True)
    regions_gdf["Name"] = pd.Categorical(regions_gdf["Name"], categories=REGION_NAMES, ordered=True)
    regions_gdf = regions_gdf.sort_values("Name").reset_index(drop=True)
    return regions_gdf

def region_mean(da, grid_ds=None, regions_gdf=None):
    """
    Mean(da) inside each region polygon.
    overlap=True lets Full Skagit coexist with sub-basins.
    """
    if regions_gdf is None:
        raise ValueError("regions_gdf is required")

    try:
        import regionmask
    except Exception as e:
        raise RuntimeError(f"regionmask required: {e}")

    time_dim = "time" if "time" in da.dims else None
    spatial = [d for d in da.dims if d != time_dim]
    if not spatial:
        return da.expand_dims(region=list(regions_gdf["Name"].values))

    host = grid_ds if grid_ds is not None else da.to_dataset(name="_tmp")
    lat_name, lon_name = find_lat_lon_names(host)
    if not (lat_name and lon_name):
        out = da.mean(spatial, skipna=True)
        return out.expand_dims(region=list(regions_gdf["Name"].values))

    lon = _to_2d(host[lon_name])
    lat = _to_2d(host[lat_name])

    regs = regionmask.Regions(
        outlines=list(regions_gdf.geometry.values),
        names=list(regions_gdf["Name"].astype(str).values),
        numbers=list(range(len(regions_gdf))),
        name="Skagit_regions",
        overlap=True,
    )

    m3 = regs.mask_3D(lon, lat)

    spatial_dims = [d for d in da.dims if d != time_dim]
    mask_spatial_dims = [d for d in m3.dims if d != "region"]

    if set(mask_spatial_dims) != set(spatial_dims):
        rename_map = {}
        for md in mask_spatial_dims:
            for pdim in spatial_dims:
                if m3.sizes.get(md) == da.sizes.get(pdim):
                    rename_map[md] = pdim
                    break
        if rename_map:
            m3 = m3.rename(rename_map)

    out_list = []
    for i, _rname in enumerate(regs.names):
        m = m3.isel(region=i)
        w = xr.where(m, 1.0, np.nan)
        out_list.append((da * w).mean(dim=spatial_dims, skipna=True))

    out = xr.concat(out_list, dim="region").assign_coords(region=regs.names)
    return out

regions_gdf = load_regions_gdf()


def to_met_year_index(idx: pd.DatetimeIndex) -> pd.Index:
    idx = pd.DatetimeIndex(idx)
    years = idx.year.to_numpy().copy()
    years[idx.month == 12] += 1
    return pd.Index(years, name="year")

def infer_step_hours(idx: pd.DatetimeIndex):
    idx = pd.DatetimeIndex(idx).sort_values()
    if len(idx) < 2:
        return np.nan
    diffs = np.diff(idx.asi8) / 1e9 / 3600.0
    diffs = diffs[diffs > 0]
    if len(diffs) == 0:
        return np.nan
    return float(np.nanmedian(diffs))

def freq_alias_from_hours(dt_hours: float):
    if not np.isfinite(dt_hours):
        return None
    if np.isclose(dt_hours, 24.0):
        return "1D"
    if np.isclose(dt_hours, round(dt_hours)):
        return f"{int(round(dt_hours))}H"
    mins = int(round(dt_hours * 60))
    return f"{mins}min"

def regularize_series(s: pd.Series) -> pd.Series:
    s = s.sort_index()
    dt_hours = infer_step_hours(s.index)
    alias = freq_alias_from_hours(dt_hours)
    if alias is None:
        return s
    start = s.index.min().floor(alias)
    end   = s.index.max().ceil(alias)
    full_idx = pd.date_range(start, end, freq=alias)
    return s.reindex(full_idx)

def filter_year_range(series: pd.Series) -> pd.Series:
    if series.empty:
        return series
    return series[(series.index >= YEAR_MIN) & (series.index <= YEAR_MAX)]

def annual_from_daily_mm(s_mm: pd.Series):
    """
    Returns:
      annual_total_in, annual_days_gt1in
    using meteorological-year labels
    """
    s_mm = s_mm.dropna().sort_index()
    if s_mm.empty:
        return pd.Series(dtype=float), pd.Series(dtype=float)

    dt_hours = infer_step_hours(s_mm.index)
    if not np.isfinite(dt_hours):
        return pd.Series(dtype=float), pd.Series(dtype=float)

    if not np.isclose(dt_hours, 24.0):
        daily_mm = s_mm.resample("1D").sum(min_count=1)
    else:
        daily_mm = s_mm.copy()

    daily_in = daily_mm / 25.4
    groups = to_met_year_index(daily_in.index)

    counts = daily_in.groupby(groups).count()
    annual_total_in = daily_in.groupby(groups).sum(min_count=1)
    annual_days_gt1in = daily_in.gt(1.0).groupby(groups).sum(min_count=1).astype(float)

    valid_years = counts[counts >= MIN_DAILY_DAYS_PER_MET_YEAR].index
    annual_total_in = annual_total_in.loc[annual_total_in.index.isin(valid_years)]
    annual_days_gt1in = annual_days_gt1in.loc[annual_days_gt1in.index.isin(valid_years)]

    annual_total_in = filter_year_range(annual_total_in)
    annual_days_gt1in = filter_year_range(annual_days_gt1in)
    return annual_total_in, annual_days_gt1in

def annual_event_max_mm(s_mm: pd.Series, duration_hours: int):
    """
    Returns annual max duration event (mm) by meteorological-year label.
    Uses rolling sum on native cadence if native cadence is <= duration.
    """
    s_mm = s_mm.dropna().sort_index()
    if s_mm.empty:
        return pd.Series(dtype=float)

    dt_hours = infer_step_hours(s_mm.index)
    if not np.isfinite(dt_hours):
        return pd.Series(dtype=float)

    if dt_hours > duration_hours + 1e-9:
        return pd.Series(dtype=float)

    ratio = duration_hours / dt_hours
    n = int(round(ratio))
    if n <= 0:
        return pd.Series(dtype=float)

    if not np.isclose(n * dt_hours, duration_hours, atol=max(1e-6, 0.05 * dt_hours)):
        return pd.Series(dtype=float)

    s_reg = regularize_series(s_mm)
    rolling_mm = s_reg.rolling(window=n, min_periods=n).sum()

    groups = to_met_year_index(rolling_mm.index)
    annual_max = rolling_mm.groupby(groups).max()

    native_groups = to_met_year_index(s_reg.index)
    counts = s_reg.groupby(native_groups).count()

    expected_steps = int(round(365.25 * 24.0 / dt_hours))
    min_steps = int(math.floor(MIN_NATIVE_COVERAGE_FRAC * expected_steps))
    valid_years = counts[counts >= min_steps].index

    annual_max = annual_max.loc[annual_max.index.isin(valid_years)]
    annual_max = filter_year_range(annual_max)
    return annual_max

def period_mean(series: pd.Series, years):
    if series.empty:
        return np.nan
    x = series[series.index.isin(years)]
    if x.empty:
        return np.nan
    return float(x.mean())

def pct_change(compare_mean, baseline_mean):
    if not np.isfinite(baseline_mean) or baseline_mean == 0 or not np.isfinite(compare_mean):
        return np.nan
    return 100.0 * (compare_mean - baseline_mean) / baseline_mean

def summarize_region_series(dataset_name, region_name, daily_s_mm, native_s_mm=None):
    """
    Returns:
      summary_row (dict),
      annual_df (DataFrame)
    """
    daily_s_mm = daily_s_mm.dropna().sort_index()
    if native_s_mm is None:
        native_s_mm = daily_s_mm.copy()
    else:
        native_s_mm = native_s_mm.dropna().sort_index()

    annual_total_in, annual_days_gt1in = annual_from_daily_mm(daily_s_mm)

    annual_max_1h  = annual_event_max_mm(native_s_mm, 1)
    annual_max_6h  = annual_event_max_mm(native_s_mm, 6)
    annual_max_24h = annual_event_max_mm(native_s_mm, 24)

    years_union = sorted(
        set(annual_total_in.index) |
        set(annual_days_gt1in.index) |
        set(annual_max_1h.index) |
        set(annual_max_6h.index) |
        set(annual_max_24h.index)
    )

    annual_rows = []
    for y in years_union:
        annual_rows.append({
            "dataset": dataset_name,
            "region": region_name,
            "year": int(y),
            "annual_total_in": float(annual_total_in.get(y, np.nan)),
            "annual_days_gt1in": float(annual_days_gt1in.get(y, np.nan)),
            "annual_max_1h_mm": float(annual_max_1h.get(y, np.nan)),
            "annual_max_6h_mm": float(annual_max_6h.get(y, np.nan)),
            "annual_max_24h_mm": float(annual_max_24h.get(y, np.nan)),
        })

    b1, c1 = period_mean(annual_max_1h, BASELINE_YEARS), period_mean(annual_max_1h, COMPARE_YEARS)
    b6, c6 = period_mean(annual_max_6h, BASELINE_YEARS), period_mean(annual_max_6h, COMPARE_YEARS)
    b24, c24 = period_mean(annual_max_24h, BASELINE_YEARS), period_mean(annual_max_24h, COMPARE_YEARS)

    summary_row = {
        "dataset": dataset_name,
        "region": region_name,
        "record_start": daily_s_mm.index.min() if not daily_s_mm.empty else pd.NaT,
        "record_end": daily_s_mm.index.max() if not daily_s_mm.empty else pd.NaT,
        "n_met_years_daily": int(len(annual_total_in)),
        "annual_avg_total_precip_in": float(annual_total_in.mean()) if not annual_total_in.empty else np.nan,
        "avg_annual_days_gt1in": float(annual_days_gt1in.mean()) if not annual_days_gt1in.empty else np.nan,
        "baseline_mean_1h_event_mm": b1,
        "compare_mean_1h_event_mm": c1,
        "pct_change_1h": pct_change(c1, b1),
        "baseline_mean_6h_event_mm": b6,
        "compare_mean_6h_event_mm": c6,
        "pct_change_6h": pct_change(c6, b6),
        "baseline_mean_24h_event_mm": b24,
        "compare_mean_24h_event_mm": c24,
        "pct_change_24h": pct_change(c24, b24),
    }

    return summary_row, pd.DataFrame(annual_rows)

def summarize_dataset_frames(dataset_name, daily_df, native_df=None):
    summary_rows = []
    annual_parts = []

    regions_present = sorted(set(daily_df["region"].unique()))
    for region in regions_present:
        daily_s = (
            daily_df.loc[daily_df["region"] == region, ["time", "mm"]]
            .dropna()
            .sort_values("time")
            .drop_duplicates(subset=["time"], keep="first")
            .set_index("time")["mm"]
        )

        native_s = None
        if native_df is not None and not native_df.empty and region in set(native_df["region"].unique()):
            native_s = (
                native_df.loc[native_df["region"] == region, ["time", "mm"]]
                .dropna()
                .sort_values("time")
                .drop_duplicates(subset=["time"], keep="first")
                .set_index("time")["mm"]
            )

        row, ann = summarize_region_series(dataset_name, region, daily_s, native_s)
        summary_rows.append(row)
        annual_parts.append(ann)

    return pd.DataFrame(summary_rows), pd.concat(annual_parts, ignore_index=True)


def _detect_prism_var(ds):
    candidates = ["ppt", "precip", "precipitation", "pr", "tp"]
    for v in candidates:
        if v in ds.data_vars:
            return v
    for v in ds.data_vars:
        lv = v.lower()
        if ("ppt" in lv) or ("prec" in lv):
            return v
    return None

def load_prism_daily(regions_gdf):
    dataset_name = "PRISM 4km"

    prism_zarrs = []
    if P_PRISM_DIR.exists():
        prism_zarrs = sorted([p for p in P_PRISM_DIR.glob("*.zarr") if p.is_dir()])

    if not prism_zarrs:
        print(f"[{dataset_name}] no zarrs found -> skip")
        return pd.DataFrame(), pd.DataFrame()

    dsets = []
    try:
        for p in prism_zarrs:
            try:
                dsets.append(safe_open_zarr(p))
            except Exception as e:
                print(f"[{dataset_name}] failed to open {p.name}: {e}")

        if not dsets:
            return pd.DataFrame(), pd.DataFrame()

        v = None
        for ds_try in dsets:
            v = _detect_prism_var(ds_try)
            if v is not None:
                break
        if v is None:
            print(f"[{dataset_name}] could not detect precip variable")
            return pd.DataFrame(), pd.DataFrame()

        da_list = []
        for ds in dsets:
            if v not in ds:
                continue
            da = ds[v]
            if "time" not in da.dims:
                tdim = next((d for d in da.dims if ("time" in d.lower()) or ("day" in d.lower())), None)
                if tdim is not None:
                    da = da.rename({tdim: "time"})
            if "time" in da.dims:
                da_list.append(da)

        if not da_list:
            print(f"[{dataset_name}] no time-aware arrays found")
            return pd.DataFrame(), pd.DataFrame()

        prism = xr.concat(da_list, dim="time", join="outer", coords="minimal", compat="override")
        prism = ensure_time_sorted_unique(prism).sel(time=slice(GLOBAL_START, GLOBAL_END))

        rm_daily = region_mean(prism, grid_ds=prism.to_dataset(name="ppt"), regions_gdf=regions_gdf)
        daily_df = da_to_region_df(rm_daily, dataset_name)
        return daily_df, daily_df.copy()

    finally:
        for ds in dsets:
            try:
                ds.close()
            except Exception:
                pass


def ensure_daymet_pkgs():
    needed = ["pystac-client", "planetary-computer", "pyproj", "fsspec", "zarr"]
    import importlib
    import subprocess
    missing = []
    for p in needed:
        mod = p.replace("-", "_")
        try:
            importlib.import_module(mod)
        except Exception:
            missing.append(p)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

ensure_daymet_pkgs()

from pystac_client import Client
import planetary_computer as pc
from pyproj import CRS, Transformer
from shapely.ops import transform as shp_transform
from fsspec.implementations.http import HTTPFileSystem

class SASAppendingHTTPFileSystem(HTTPFileSystem):
    """
    Appends the SAS query string to every blob request correctly, so requests become:
      .../na.zarr/.zgroup?<sas>
    instead of the broken:
      .../na.zarr?<sas>/.zgroup
    """
    def __init__(self, query: str, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._fixed_query = query.lstrip("?")

    def _with_query(self, url: str) -> str:
        u = urlparse(url)
        q = u.query or self._fixed_query
        return urlunparse((u.scheme, u.netloc, u.path, u.params, q, u.fragment))

    def _open(self, path, mode="rb", block_size=None, **kwargs):
        return super()._open(self._with_query(path), mode=mode, block_size=block_size, **kwargs)

    async def _cat_file(self, url, start=None, end=None, **kwargs):
        return await super()._cat_file(self._with_query(url), start=start, end=end, **kwargs)

    async def _info(self, url, **kwargs):
        return await super()._info(self._with_query(url), **kwargs)

    async def _isfile(self, url, **kwargs):
        return await super()._isfile(self._with_query(url), **kwargs)

    async def _exists(self, url, **kwargs):
        return await super()._exists(self._with_query(url), **kwargs)

def get_daymet_mapper():
    """
    Returns an fsspec mapper where the base path is clean:
      https://.../na.zarr
    and the SAS token is injected by the custom filesystem on each object request.
    """
    stac = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
    col = stac.get_collection("daymet-daily-na")

    asset_key = "zarr-https" if "zarr-https" in col.assets else "zarr-abfs"
    signed = pc.sign(col.assets[asset_key]).href

    u = urlparse(signed)
    base = f"{u.scheme}://{u.netloc}{u.path}"
    query = u.query

    fs = SASAppendingHTTPFileSystem(query=query)
    return fs.get_mapper(base)

def safe_open_daymet_zarr():
    """
    Fresh signed mapper + robust consolidated fallback.
    """
    mapper = get_daymet_mapper()
    try:
        return xr.open_zarr(mapper, consolidated=True)
    except Exception:
        mapper = get_daymet_mapper()
        return xr.open_zarr(mapper, consolidated=False)

def daymet_lcc_crs():
    return CRS.from_proj4(
        "+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 "
        "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
    )

_DAYMET_TFM_LL_TO_XY = Transformer.from_crs("EPSG:4326", daymet_lcc_crs(), always_xy=True)

def lonlat_to_daymet_xy_bounds(geom_wgs84, pad_m=5000.0):
    minx, miny, maxx, maxy = geom_wgs84.bounds
    corners = [(minx, miny), (minx, maxy), (maxx, miny), (maxx, maxy)]
    xs, ys = zip(*[_DAYMET_TFM_LL_TO_XY.transform(lon, lat) for lon, lat in corners])
    return (
        float(min(xs) - pad_m),
        float(max(xs) + pad_m),
        float(min(ys) - pad_m),
        float(max(ys) + pad_m),
    )

def _daymet_project_geom_to_xy(geom_ll):
    def _proj(x, y, z=None):
        return _DAYMET_TFM_LL_TO_XY.transform(x, y)
    return shp_transform(_proj, geom_ll)

_DAYMET_MASK_CACHE = {}

def load_daymet_daily(regions_gdf):
    dataset_name = "Daymet-PC"

    ds = None
    try:
        ds = safe_open_daymet_zarr()

        if "prcp" not in ds:
            print(f"[{dataset_name}] missing 'prcp' variable")
            return pd.DataFrame(), pd.DataFrame()

        regs = regions_gdf.to_crs("EPSG:4326")
        parts = []

        for rname in REGION_NAMES:
            geom_ll = regs.loc[regs["Name"] == rname, "geometry"].iloc[0]

            xmin, xmax, ymin, ymax = lonlat_to_daymet_xy_bounds(geom_ll, pad_m=5000.0)

            y0 = float(ds["y"].values[0])
            y1 = float(ds["y"].values[-1])
            y_slice = slice(ymax, ymin) if y0 > y1 else slice(ymin, ymax)

            pr = ds["prcp"].sel(
                time=slice(GLOBAL_START, GLOBAL_END),
                x=slice(xmin, xmax),
                y=y_slice,
            ).chunk({"time": 365, "y": 256, "x": 256})

            key = (
                float(pr["x"].values[0]), float(pr["x"].values[-1]),
                float(pr["y"].values[0]), float(pr["y"].values[-1]),
                pr.sizes["x"], pr.sizes["y"], rname
            )

            mask = _DAYMET_MASK_CACHE.get(key)
            if mask is None:
                xx, yy = xr.broadcast(pr["x"], pr["y"])
                geom_xy = _daymet_project_geom_to_xy(geom_ll)

                from shapely import contains_xy
                mask_np = contains_xy(geom_xy, np.asarray(xx.values), np.asarray(yy.values))
                mask = xr.DataArray(mask_np, dims=xx.dims, coords=xx.coords)
                _DAYMET_MASK_CACHE[key] = mask

            ts = pr.where(mask).mean(dim=("y", "x"), skipna=True).compute()

            df = ts.to_dataframe(name="mm").reset_index()[["time", "mm"]]
            df["time"] = pd.to_datetime(df["time"])
            df["region"] = rname
            df["dataset"] = dataset_name
            parts.append(df[["dataset", "region", "time", "mm"]])

        if not parts:
            return pd.DataFrame(), pd.DataFrame()

        daily_df = (
            pd.concat(parts, ignore_index=True)
              .sort_values(["region", "time"])
              .drop_duplicates(subset=["region", "time"], keep="first")
              .reset_index(drop=True)
        )

        return daily_df, daily_df.copy()

    except Exception as e:
        print(f"[{dataset_name}] failed: {type(e).__name__}: {e}")
        return pd.DataFrame(), pd.DataFrame()

    finally:
        if ds is not None:
            try:
                ds.close()
            except Exception:
                pass


def _find_hrrr_tp_var(ds):
    if "tp" in ds.data_vars:
        return "tp"
    if "apcp" in ds.data_vars:
        return "apcp"
    for v in ds.data_vars:
        lv = v.lower()
        if lv == "tp" or "apcp" in lv or "prec" in lv:
            return v
    return None

def load_hrrr_f06(regions_gdf):
    dataset_name = "HRRR F06"

    if not P_HRRR_DIR.exists():
        print(f"[{dataset_name}] directory missing -> skip")
        return pd.DataFrame(), pd.DataFrame()

    zarrs = sorted([p for p in P_HRRR_DIR.glob("*.zarr") if p.is_dir()])
    monthly = [p for p in zarrs if re.match(r"\d{4}-\d{2}_HRRR_f06_data\.zarr", p.name)]

    if not monthly:
        print(f"[{dataset_name}] no monthly f06 zarrs found -> skip")
        return pd.DataFrame(), pd.DataFrame()

    native_pieces = []

    for p in monthly:
        ds = None
        try:
            ds = safe_open_zarr(p)
            var = _find_hrrr_tp_var(ds)
            if var is None:
                print(f"[{dataset_name}] no precip var in {p.name}")
                continue

            da = ensure_time_sorted_unique(ds[var])

            t = pd.to_datetime(da["time"].values)
            keep = np.isin(t.hour, [0, 6, 12, 18])
            da = da.sel(time=da["time"].values[keep])

            da = da.assign_coords(time=(pd.to_datetime(da["time"].values) + pd.Timedelta(hours=6)))
            da = da.sel(time=slice(GLOBAL_START, GLOBAL_END))

            if da.sizes.get("time", 0) == 0:
                continue

            rm_native = region_mean(da, grid_ds=ds, regions_gdf=regions_gdf)
            native_pieces.append(rm_native)

        except Exception as e:
            print(f"[{dataset_name}] {p.name} failed: {type(e).__name__}: {e}")
        finally:
            if ds is not None:
                try:
                    ds.close()
                except Exception:
                    pass

    if not native_pieces:
        return pd.DataFrame(), pd.DataFrame()

    native = xr.concat(native_pieces, dim="time", join="outer", coords="minimal", compat="override")
    native = ensure_time_sorted_unique(native).sel(time=slice(GLOBAL_START, GLOBAL_END))
    daily = native.resample(time="1D").sum()

    native_df = da_to_region_df(native, dataset_name)
    daily_df  = da_to_region_df(daily, dataset_name)
    return daily_df, native_df


def pick_time_dim(da):
    if "time" in da.dims:
        return "time"
    if "day" in da.dims:
        return "day"
    for d in da.dims:
        if "time" in d.lower() or "day" in d.lower():
            return d
    raise ValueError(f"[UCLA] No time-like dim found. dims={da.dims}")

def ensure_sorted_unique_time(da, tdim="time"):
    da = da.sortby(tdim)
    try:
        t = pd.to_datetime(da[tdim].values)
        _, idx = np.unique(t, return_index=True)
        da = da.isel({tdim: np.sort(idx)})
    except Exception:
        pass
    return da

def build_ucla_region_masks(coord_file: Path, regions_gdf):
    import shapely
    try:
        from shapely import contains_xy
        has_contains_xy = True
    except Exception:
        has_contains_xy = False
        from shapely.prepared import prep

    g = xr.open_dataset(coord_file, engine="netcdf4")
    lat = g["lat2d"].squeeze(drop=True).astype("float64")
    lon = g["lon2d"].squeeze(drop=True).astype("float64")
    g.close()

    lonv = lon.values.ravel()
    latv = lat.values.ravel()
    ny, nx = lon.shape

    masks = {}
    for _, row in regions_gdf.iterrows():
        name = str(row["Name"])
        geom = row["geometry"]

        if has_contains_xy:
            m_flat = contains_xy(geom, lonv, latv)
        else:
            pg = prep(geom)
            m_flat = np.array([pg.contains(shapely.Point(x, y)) for x, y in zip(lonv, latv)], dtype=bool)

        mask2d = m_flat.reshape(ny, nx).astype("float32")
        masks[name] = xr.DataArray(mask2d, dims=lon.dims, coords=lon.coords)

    return masks

def masked_mean_timeseries(pr: xr.DataArray, mask2d: xr.DataArray) -> xr.DataArray:
    spatial_dims = [d for d in pr.dims if d != "time"]

    mask = mask2d
    if set(mask.dims) != set(spatial_dims):
        rename_map = {}
        for md in mask.dims:
            for pdim in spatial_dims:
                if mask.sizes[md] == pr.sizes[pdim]:
                    rename_map[md] = pdim
                    break
        mask = mask.rename(rename_map)

    denom = mask.sum(dim=spatial_dims, skipna=True)
    if float(denom.values) == 0.0:
        return xr.full_like(pr.isel(time=0), np.nan).expand_dims({"time": pr["time"]})

    pr = pr.chunk({"time": min(366, pr.sizes["time"])})
    num = (pr * mask).sum(dim=spatial_dims, skipna=True)
    return num / denom

def load_ucla_daily(regions_gdf):
    dataset_name = "UCLA ERA5 d02"

    if not UCLA_PREC_DIR.exists() or not UCLA_COORD_FILE.exists():
        print(f"[{dataset_name}] missing precip dir or coord file -> skip")
        return pd.DataFrame(), pd.DataFrame()

    masks = build_ucla_region_masks(UCLA_COORD_FILE, regions_gdf)
    files = sorted(glob(str(UCLA_PREC_DIR / "prec.daily.era5.d02.*.nc")))
    if not files:
        print(f"[{dataset_name}] no files found -> skip")
        return pd.DataFrame(), pd.DataFrame()

    parts = []
    for f in files:
        ds = None
        try:
            ds = xr.open_dataset(f, engine="netcdf4", decode_cf=True, mask_and_scale=True)
            if "prec" not in ds:
                continue

            pr = ds["prec"]
            tdim = pick_time_dim(pr)
            if tdim != "time":
                pr = pr.rename({tdim: "time"})

            pr = ensure_sorted_unique_time(pr, "time").sel(time=slice(GLOBAL_START, GLOBAL_END))
            if pr.sizes.get("time", 0) == 0:
                continue

            for rname in REGION_NAMES:
                ts = masked_mean_timeseries(pr, masks[rname]).compute()
                df = ts.to_dataframe(name="mm").reset_index()[["time", "mm"]]
                df["time"] = pd.to_datetime(df["time"])
                df["region"] = rname
                df["dataset"] = dataset_name
                parts.append(df[["dataset", "region", "time", "mm"]])

        except Exception as e:
            print(f"[{dataset_name}] {Path(f).name} failed: {type(e).__name__}: {e}")
        finally:
            if ds is not None:
                try:
                    ds.close()
                except Exception:
                    pass

    if not parts:
        return pd.DataFrame(), pd.DataFrame()

    daily_df = (
        pd.concat(parts, ignore_index=True)
        .sort_values(["region", "time"])
        .drop_duplicates(subset=["region", "time"], keep="first")
        .reset_index(drop=True)
    )
    return daily_df, daily_df.copy()


def load_snotel_daily(regions_gdf):
    dataset_name = "SNOTEL (mean stations)"

    try:
        import geopandas as gpd
        import shapely.geometry as sgeom
        import requests
    except Exception as e:
        print(f"[{dataset_name}] missing dependency: {e}")
        return pd.DataFrame(), pd.DataFrame()

    stations_url = "https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/all_stations.geojson"
    try:
        r = requests.get(stations_url, timeout=30)
        r.raise_for_status()
        gj = r.json()
    except Exception as e:
        print(f"[{dataset_name}] failed to download stations metadata: {e}")
        return pd.DataFrame(), pd.DataFrame()

    records, geoms = [], []
    for feat in gj.get("features", []):
        props = feat.get("properties", {})
        geom = feat.get("geometry")
        if geom is None:
            continue
        records.append(props)
        geoms.append(sgeom.shape(geom))

    if not records:
        return pd.DataFrame(), pd.DataFrame()

    stations = gpd.GeoDataFrame(records, geometry=geoms, crs="EPSG:4326")
    skagit_geom = gpd.read_file(BOUNDARY_GEO).to_crs("EPSG:4326").geometry.iloc[0]
    inside = stations.geometry.within(skagit_geom)
    skagit_stations = stations[inside].copy()

    desired_names = {"Beaver Pass", "Brown Top", "Marten Ridge", "Rainy Pass", "Swamp Creek", "Thunder Basin"}
    if "name" in skagit_stations.columns:
        skagit_stations = skagit_stations[skagit_stations["name"].isin(desired_names)]

    if skagit_stations.empty:
        print(f"[{dataset_name}] no stations found after filtering")
        return pd.DataFrame(), pd.DataFrame()

    regs3 = regions_gdf[regions_gdf["Name"].isin(["Upper Skagit", "Sauk", "Lower Skagit"])].copy().to_crs("EPSG:4326")
    try:
        joined = gpd.sjoin(skagit_stations, regs3.rename(columns={"Name": "region"}), predicate="within", how="left")
    except TypeError:
        joined = gpd.sjoin(skagit_stations, regs3.rename(columns={"Name": "region"}), op="within", how="left")

    station_region_map = {}
    for _, row in joined.dropna(subset=["region"]).iterrows():
        code = row.get("code")
        region = row.get("region")
        if pd.notna(code):
            station_region_map[str(code)] = str(region)

    base_csv_url = "https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/data"
    station_series = {}

    for _, st in skagit_stations.iterrows():
        code = st.get("code")
        if pd.isna(code):
            continue
        code = str(code)
        csv_url = f"{base_csv_url}/{code}.csv"
        try:
            df = pd.read_csv(csv_url, index_col="datetime", parse_dates=True)
        except Exception as e:
            print(f"[{dataset_name}] failed to read {code}: {e}")
            continue

        if "PRCPSA" not in df.columns:
            continue

        s = (df["PRCPSA"] * 1000.0).sort_index()
        s = s.loc[GLOBAL_START:GLOBAL_END]

        dt = infer_step_hours(pd.DatetimeIndex(s.index))
        if np.isfinite(dt) and dt < 24:
            s = s.resample("1D").sum(min_count=1)

        station_series[code] = s

    if not station_series:
        return pd.DataFrame(), pd.DataFrame()

    def mean_series(codes):
        use = [station_series[c] for c in codes if c in station_series]
        if not use:
            return pd.Series(dtype=float)
        mat = pd.concat(use, axis=1)
        return mat.mean(axis=1, skipna=True).sort_index()

    region_codes = {
        region: [code for code, reg in station_region_map.items() if reg == region]
        for region in ["Upper Skagit", "Sauk", "Lower Skagit"]
    }
    full_codes = list(station_series.keys())

    parts = []
    for region in ["Upper Skagit", "Sauk", "Lower Skagit"]:
        s = mean_series(region_codes.get(region, []))
        if s.empty:
            continue
        parts.append(series_to_time_df(s, dataset_name, region))

    s_full = mean_series(full_codes)
    if not s_full.empty:
        parts.append(series_to_time_df(s_full, dataset_name, "Full Skagit"))

    if not parts:
        return pd.DataFrame(), pd.DataFrame()

    daily_df = (
        pd.concat(parts, ignore_index=True)
          .sort_values(["region", "time"])
          .drop_duplicates(subset=["region", "time"], keep="first")
          .reset_index(drop=True)
    )
    return daily_df, daily_df.copy()


def pnnl_region_mean_with_mask(p, mask_ds):
    """
    Fallback if regionmask-on-native-grid fails.
    Usually yields only the 3 sub-basins present in the mask file.
    """
    if "mask" not in mask_ds:
        raise ValueError("[PNNL] Expected variable 'mask' in fallback region mask")

    region_labels = [str(r) for r in mask_ds["region"].values]
    spatial_dims = [d for d in p.dims if d != "time"]

    out_list = []
    out_regions = []

    for rname in region_labels:
        mr = mask_ds["mask"].sel(region=rname)
        if set(mr.dims) != set(spatial_dims):
            rename_map = {}
            for md in mr.dims:
                for pdim in spatial_dims:
                    if mr.sizes[md] == p.sizes[pdim]:
                        rename_map[md] = pdim
                        break
            if rename_map:
                mr = mr.rename(rename_map)

        out_list.append(p.where(mr).mean(dim=spatial_dims, skipna=True))
        out_regions.append(rname)

    out = xr.concat(out_list, dim="region").assign_coords(region=out_regions)
    return out

def load_pnnl_native(regions_gdf):
    dataset_name = "PNNL hist"

    if not PNNL_HIST.exists():
        print(f"[{dataset_name}] directory missing -> skip")
        return pd.DataFrame(), pd.DataFrame()

    mask_ds = None
    if PNNL_REGION_MASK.exists():
        try:
            mask_ds = xr.open_dataset(PNNL_REGION_MASK)
        except Exception:
            mask_ds = None

    native_pieces = []

    year_dirs = sorted([p for p in PNNL_HIST.glob("*") if p.is_dir() and p.name.isdigit()])
    if not year_dirs:
        print(f"[{dataset_name}] no yearly directories found")
        return pd.DataFrame(), pd.DataFrame()

    for yd in year_dirs:
        yy = int(yd.name)
        files = sorted(glob(str(yd / "*PREC_ACC_NC*.nc")))
        if not files:
            continue

        def _preprocess(ds):
            keep = [v for v in ds.data_vars if "PREC_ACC_NC" in v]
            return ds[keep] if keep else ds

        ds = None
        try:
            ds = xr.open_mfdataset(
                files,
                combine="by_coords",
                engine="netcdf4",
                parallel=False,
                preprocess=_preprocess,
                chunks={"time": 24 * 31},
            )

            v = next((vv for vv in ds.data_vars if "PREC_ACC_NC" in vv), None)
            if v is None:
                continue

            p = ensure_time_sorted_unique(ds[v]).sel(time=slice(GLOBAL_START, GLOBAL_END))
            if p.sizes.get("time", 0) == 0:
                continue

            try:
                rm_native = region_mean(p, grid_ds=ds, regions_gdf=regions_gdf)
            except Exception as e1:
                if mask_ds is not None:
                    print(f"[{dataset_name}] regionmask failed for {yy}, using fallback mask ({type(e1).__name__})")
                    rm_native = pnnl_region_mean_with_mask(p, mask_ds)
                else:
                    raise e1

            native_pieces.append(rm_native)

        except Exception as e:
            print(f"[{dataset_name}] year {yy} failed: {type(e).__name__}: {e}")
        finally:
            if ds is not None:
                try:
                    ds.close()
                except Exception:
                    pass

    if mask_ds is not None:
        try:
            mask_ds.close()
        except Exception:
            pass

    if not native_pieces:
        return pd.DataFrame(), pd.DataFrame()

    native = xr.concat(native_pieces, dim="time", join="outer", coords="minimal", compat="override")
    native = ensure_time_sorted_unique(native).sel(time=slice(GLOBAL_START, GLOBAL_END))
    daily = native.resample(time="1D").sum()

    native_df = da_to_region_df(native, dataset_name)
    daily_df  = da_to_region_df(daily, dataset_name)
    return daily_df, native_df


def load_conus404_daily(regions_gdf):
    dataset_name = "CONUS404"

    try:
        import intake
        from shapely import contains_xy
    except Exception as e:
        print(f"[{dataset_name}] missing dependency: {e}")
        return pd.DataFrame(), pd.DataFrame()

    url = "https://raw.githubusercontent.com/hytest-org/hytest/main/dataset_catalog/hytest_intake_catalog.yml"

    try:
        cat = intake.open_catalog(url)
        ds = cat["conus404-catalog"]["conus404-daily-osn"].to_dask()
    except Exception as e:
        print(f"[{dataset_name}] failed to open intake catalog: {e}")
        return pd.DataFrame(), pd.DataFrame()

    if "PREC_ACC_NC" not in ds.data_vars:
        print(f"[{dataset_name}] missing PREC_ACC_NC")
        return pd.DataFrame(), pd.DataFrame()

    precip = ds["PREC_ACC_NC"].sel(time=slice(GLOBAL_START, GLOBAL_END))
    lat = ds["lat"]
    lon = ds["lon"]

    parts = []
    for region in REGION_NAMES:
        try:
            geom = regions_gdf.loc[regions_gdf["Name"] == region, "geometry"].iloc[0]
            mask_np = contains_xy(geom, lon.values, lat.values)
            mask_da = xr.DataArray(mask_np, dims=lat.dims, coords=lat.coords)

            ts = precip.where(mask_da).mean(dim=("y", "x"), skipna=True).compute()
            df = ts.to_dataframe(name="mm").reset_index()[["time", "mm"]]
            df["time"] = pd.to_datetime(df["time"])
            df["region"] = region
            df["dataset"] = dataset_name
            parts.append(df[["dataset", "region", "time", "mm"]])
        except Exception as e:
            print(f"[{dataset_name}] region {region} failed: {type(e).__name__}: {e}")

    if not parts:
        return pd.DataFrame(), pd.DataFrame()

    daily_df = pd.concat(parts, ignore_index=True).sort_values(["region", "time"]).reset_index(drop=True)
    return daily_df, daily_df.copy()


all_summary = []
all_annual = []

def run_one(loader_name, loader_func):
    print(f"\n{'='*70}\nRunning {loader_name}\n{'='*70}")
    t0 = time.time()
    daily_df, native_df = loader_func(regions_gdf)
    if daily_df is None or daily_df.empty:
        print(f"[{loader_name}] no data returned")
        return
    summary_df, annual_df = summarize_dataset_frames(loader_name, daily_df, native_df)
    all_summary.append(summary_df)
    all_annual.append(annual_df)
    print(f"[{loader_name}] done in {(time.time() - t0)/60.0:.2f} min")

if RUN_DATASETS["PRISM_4KM"]:
    run_one("PRISM 4km", load_prism_daily)

if RUN_DATASETS["DAYMET_PC"]:
    run_one("Daymet-PC", load_daymet_daily)

if RUN_DATASETS["HRRR_F06"]:
    run_one("HRRR F06", load_hrrr_f06)

if RUN_DATASETS["UCLA_ERA5_D02"]:
    run_one("UCLA ERA5 d02", load_ucla_daily)

if RUN_DATASETS["PNNL_HIST"]:
    run_one("PNNL hist", load_pnnl_native)

if RUN_DATASETS["CONUS404"]:
    run_one("CONUS404", load_conus404_daily)

if RUN_DATASETS["SNOTEL"]:
    run_one("SNOTEL (mean stations)", load_snotel_daily)

if not all_summary:
    raise RuntimeError("No dataset produced output.")

summary_df = pd.concat(all_summary, ignore_index=True)
annual_df  = pd.concat(all_annual, ignore_index=True)


annual_df = annual_df.sort_values(["region", "dataset", "year"]).reset_index(drop=True)
summary_df = summary_df.sort_values(["region", "dataset"]).reset_index(drop=True)

annual_df.to_csv(ANNUAL_CSV, index=False)
summary_df.to_csv(SUMMARY_CSV, index=False)

print("Saved annual metrics ->", ANNUAL_CSV)
print("Saved summary metrics ->", SUMMARY_CSV)


metric_name_map = {
    "annual_avg_total_precip_in": "Annual Average Total Precipitation (in)",
    "avg_annual_days_gt1in": 'Average Annual Number of Days with >1" Total Precipitation',
    "pct_change_1h": "Average Percent Change in 1 Hour Duration Precipitation Event",
    "pct_change_6h": "Average Percent Change in 6 Hour Duration Precipitation Event",
    "pct_change_24h": "Average Percent Change in 24 Hour Duration Precipitation Event",
}

metric_order = [
    "annual_avg_total_precip_in",
    "avg_annual_days_gt1in",
    "pct_change_1h",
    "pct_change_6h",
    "pct_change_24h",
]

for region in REGION_NAMES:
    sub = summary_df[summary_df["region"] == region].copy()
    if sub.empty:
        continue

    wide = sub.set_index("dataset")[metric_order].T
    wide.index = [metric_name_map[x] for x in wide.index]

    print("\n" + "="*100)
    print(f"REGION: {region}")
    print("="*100)
    display(wide.round(2))


event_debug_cols = [
    "dataset", "region",
    "baseline_mean_1h_event_mm", "compare_mean_1h_event_mm", "pct_change_1h",
    "baseline_mean_6h_event_mm", "compare_mean_6h_event_mm", "pct_change_6h",
    "baseline_mean_24h_event_mm", "compare_mean_24h_event_mm", "pct_change_24h",
]
display(summary_df[event_debug_cols].round(2))


for region in REGION_NAMES:
    sub = summary_df[summary_df["region"] == region].copy()
    if sub.empty:
        continue

    wide = sub.set_index("dataset")[metric_order].T
    wide.index = [metric_name_map[x] for x in wide.index]

    out_csv = OUT / f"summary_table_{region.replace(' ', '_')}_{YEAR_MIN}_{YEAR_MAX}.csv"
    wide.to_csv(out_csv)
    print("Saved region summary table ->", out_csv)


Running PRISM 4km
[PRISM 4km] done in 0.10 min

Running Daymet-PC
[Daymet-PC] failed: TimeoutError: 
[Daymet-PC] no data returned

Running HRRR F06


Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7e0a188a7980>


[HRRR F06] done in 2.24 min

Running UCLA ERA5 d02
[UCLA ERA5 d02] done in 0.90 min

Running PNNL hist
[PNNL hist] done in 15.42 min

Running CONUS404
[CONUS404] region Full Skagit failed: RuntimeError: Zstd decompression error: b'Data corruption detected'
[CONUS404] done in 132.82 min

Running SNOTEL (mean stations)
[SNOTEL (mean stations)] done in 0.05 min
Saved annual metrics -> /data0/balaji24/data/derived/precip_summary_metrics/precip_annual_metrics_1983_2024.csv
Saved summary metrics -> /data0/balaji24/data/derived/precip_summary_metrics/precip_summary_metrics_1983_2024.csv

REGION: Upper Skagit


dataset,CONUS404,HRRR F06,PNNL hist,PRISM 4km,SNOTEL (mean stations),UCLA ERA5 d02
Annual Average Total Precipitation (in),86.17,66.61,21.67,87.96,80.92,96.82
"Average Annual Number of Days with >1"" Total Precipitation",23.33,16.60,0.00,21.90,17.69,26.64
Average Percent Change in 1 Hour Duration Precipitation Event,NaN,NaN,-2.87,NaN,NaN,NaN
Average Percent Change in 6 Hour Duration Precipitation Event,NaN,NaN,-4.86,NaN,NaN,NaN
Average Percent Change in 24 Hour Duration Precipitation Event,-7.34,NaN,-8.54,-12.93,14.47,-5.82



REGION: Sauk


...

dataset,CONUS404,HRRR F06,PNNL hist,PRISM 4km,UCLA ERA5 d02
Annual Average Total Precipitation (in),69.75,66.45,21.67,92.15,73.30
"Average Annual Number of Days with >1"" Total Precipitation",16.38,15.60,0.00,22.93,16.45
Average Percent Change in 1 Hour Duration Precipitation Event,NaN,NaN,-2.87,NaN,NaN
Average Percent Change in 6 Hour Duration Precipitation Event,NaN,NaN,-4.86,NaN,NaN
Average Percent Change in 24 Hour Duration Precipitation Event,-5.65,NaN,-8.54,-15.49,-3.90



REGION: Full Skagit


dataset,HRRR F06,PNNL hist,PRISM 4km,SNOTEL (mean stations),UCLA ERA5 d02
Annual Average Total Precipitation (in),69.25,21.67,92.91,80.92,96.09
"Average Annual Number of Days with >1"" Total Precipitation",17.00,0.00,23.00,17.69,26.12
Average Percent Change in 1 Hour Duration Precipitation Event,NaN,-2.87,NaN,NaN,NaN
Average Percent Change in 6 Hour Duration Precipitation Event,NaN,-4.86,NaN,NaN,NaN
Average Percent Change in 24 Hour Duration Precipitation Event,NaN,-8.54,-11.04,14.47,-4.52


,dataset,region,baseline_mean_1h_event_mm,compare_mean_1h_event_mm,pct_change_1h,baseline_mean_6h_event_mm,compare_mean_6h_event_mm,pct_change_6h,baseline_mean_24h_event_mm,compare_mean_24h_event_mm,pct_change_24h
0,HRRR F06,Full Skagit,NaN,NaN,NaN,NaN,27.13,NaN,NaN,76.52,NaN
1,PNNL hist,Full Skagit,0.32,0.31,-2.87,1.76,1.68,-4.86,5.88,5.38,-8.54
2,PRISM 4km,Full Skagit,NaN,NaN,NaN,NaN,NaN,NaN,85.30,75.88,-11.04
3,SNOTEL (mean stations),Full Skagit,NaN,NaN,NaN,NaN,NaN,NaN,77.32,88.51,14.47
4,UCLA ERA5 d02,Full Skagit,NaN,NaN,NaN,NaN,NaN,NaN,79.33,75.74,-4.52
5,CONUS404,Lower Skagit,NaN,NaN,NaN,NaN,NaN,NaN,63.53,59.95,-5.65
6,HRRR F06,Lower Skagit,NaN,NaN,NaN,NaN,27.10,NaN,NaN,70.33,NaN
7,PNNL hist,Lower Skagit,0.32,0.31,-2.87,1.76,1.68,-4.86,5.88,5.38,-8.54
8,PRISM 4km,Lower Skagit,NaN,NaN,NaN,NaN,NaN,NaN,82.45,69.68,-15.49
9,UCLA ERA5 d02,Lower Skagit,NaN,NaN,NaN,NaN,NaN,NaN,67.28,64.66,-3.90


Saved region summary table -> /data0/balaji24/data/derived/precip_summary_metrics/summary_table_Upper_Skagit_1983_2024.csv
Saved region summary table -> /data0/balaji24/data/derived/precip_summary_metrics/summary_table_Sauk_1983_2024.csv
Saved region summary table -> /data0/balaji24/data/derived/precip_summary_metrics/summary_table_Lower_Skagit_1983_2024.csv
Saved region summary table -> /data0/balaji24/data/derived/precip_summary_metrics/summary_table_Full_Skagit_1983_2024.csv


In [ ]:
# ============================================================
# NOTEBOOK: Skagit precipitation summary metrics across datasets
#
# Computes, for each dataset x region:
#   1) Annual Average Total Precipitation (mm)
#   2) Average Annual Number of Days with >25.4 mm Total Precipitation
#   3) Average Percent Change in 1h / 6h / 24h precipitation event
#
# DEFINITION USED HERE FOR PERCENT CHANGE:
#   % change = 100 * (compare_mean - baseline_mean) / baseline_mean
#   where compare_mean and baseline_mean are based on the
#   mean annual MAXIMUM event magnitude for that duration.
#
# YEAR CONVENTION:
#   meteorological-year label Y = Dec(Y-1) + Jan..Nov(Y)
#
# IMPORTANT:
# - Daily-only datasets -> 1h, 6h are NaN
# - HRRR F06 -> 6h and 24h work, 1h is NaN
# - PNNL hourly -> 1h, 6h, 24h should work
# - SNOTEL depends on station CSV cadence; this notebook treats it as daily
# - PNNL Full Skagit may be unavailable if fallback mask is used and only
#   contains 3 sub-basins
# ============================================================


import re
import sys
import time
import math
import warnings
from glob import glob
from pathlib import Path
from urllib.parse import urlparse, urlunparse

import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- Config ----------------
BASE = Path("/data0/balaji24/data")
BOUNDARY_GEO = Path("../data/GIS/SkagitBoundary.json")
HUC8_GEO     = Path("../data/GIS/SkagitSubBasin_HUC8.geojson")

P_PRISM_DIR  = BASE / "prism_ppt"
P_HRRR_DIR   = BASE / "weather_data_06"
PNNL_HIST    = BASE / "PNNL" / "historical"

UCLA_PREC_DIR   = Path("/data0/balaji24/data/ucla_era5_d02_daily/prec")
UCLA_COORD_FILE = Path("/data0/balaji24/data/ucla_era5_d02_daily/static/wrfinput_d02_coord.nc")

# optional fallback only if PNNL regionmask directly on native grid fails
PNNL_REGION_MASK = Path("/home/balaji24/skagit-met/analysis/skagit_huc8_3mask_pnnl_450x450.nc")

OUT = BASE / "derived" / "precip_summary_metrics"
OUT.mkdir(parents=True, exist_ok=True)

REGION_NAMES = ["Upper Skagit", "Sauk", "Lower Skagit", "Full Skagit"]

YEAR_MIN = 1983
YEAR_MAX = 2024

GLOBAL_START = f"{YEAR_MIN-1}-12-01"
GLOBAL_END   = f"{YEAR_MAX}-11-30"

# ----- Choose the 2 periods for percent-change metrics -----
BASELINE_YEARS = list(range(1985, 2000))   # 1985..1999
COMPARE_YEARS  = list(range(2010, 2025))   # 2010..2024

# Coverage rules
MIN_DAILY_DAYS_PER_MET_YEAR = 330
MIN_NATIVE_COVERAGE_FRAC    = 0.90

# Threshold now in mm instead of inches
GT_DAILY_THRESH_MM = 25.4

# Daymet robustness
DAYMET_MAX_TRIES = 4
DAYMET_SLEEP_BASE = 15

# Turn datasets on/off
RUN_DATASETS = {
    "PRISM_4KM": True,
    "DAYMET_PC": True,
    "HRRR_F06": True,
    "UCLA_ERA5_D02": True,
    "PNNL_HIST": True,
    "CONUS404": True,
    "SNOTEL": True,
}

# output files
ANNUAL_CSV  = OUT / f"precip_annual_metrics_{YEAR_MIN}_{YEAR_MAX}.csv"
SUMMARY_CSV = OUT / f"precip_summary_metrics_{YEAR_MIN}_{YEAR_MAX}.csv"


def safe_open_zarr(p: Path):
    try:
        return xr.open_zarr(p, consolidated=True)
    except Exception:
        return xr.open_zarr(p, consolidated=False)

def ensure_time_sorted_unique(da):
    if "time" not in da.dims:
        return da
    da = da.sortby("time")
    try:
        t = pd.to_datetime(da["time"].values)
        _, idx = np.unique(t, return_index=True)
        da = da.isel(time=np.sort(idx))
    except Exception:
        pass
    return da

def find_lat_lon_names(ds):
    lat_name = next((n for n in ["lat","latitude","XLAT","XLAT_M","lat2d"] if n in ds), None)
    lon_name = next((n for n in ["lon","longitude","XLONG","XLONG_M","lon2d"] if n in ds), None)
    return lat_name, lon_name

def _to_2d(arr):
    if "time" in arr.dims:
        arr = arr.isel(time=0)
    for d in list(arr.dims):
        if arr.sizes[d] == 1:
            arr = arr.isel({d: 0})
    while arr.ndim > 2:
        extra = [d for d in arr.dims if d not in ("x","y","lon","lat")]
        if not extra:
            break
        arr = arr.isel({extra[0]: 0})
    return arr

def da_to_region_df(da, dataset_name):
    """
    DataArray dims expected: time, region
    Returns tidy df: dataset, region, time, mm
    """
    da = ensure_time_sorted_unique(da)
    df = da.to_dataframe(name="mm").reset_index()
    need = ["time", "region", "mm"]
    df = df[[c for c in need if c in df.columns]].copy()
    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"])
    df["dataset"] = dataset_name
    df = df[["dataset", "region", "time", "mm"]].sort_values(["region", "time"]).reset_index(drop=True)
    return df

def series_to_time_df(s: pd.Series, dataset_name: str, region_name: str) -> pd.DataFrame:
    """
    Convert a time-indexed Series -> DataFrame with columns:
    dataset, region, time, mm
    Works even if the index name is 'datetime', 'index', or anything else.
    """
    if s is None or len(s) == 0:
        return pd.DataFrame(columns=["dataset", "region", "time", "mm"])

    s = s.sort_index()
    df = s.rename("mm").to_frame().reset_index()
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "time"})
    df["time"] = pd.to_datetime(df["time"])
    df["region"] = region_name
    df["dataset"] = dataset_name
    return df[["dataset", "region", "time", "mm"]]

def normalize_precip_to_mm(da: xr.DataArray, source_name: str = "", verbose: bool = False) -> xr.DataArray:
    """
    Normalize precipitation to mm using attrs['units'] when available.
    If units are unknown, leaves values unchanged.
    """
    units = str(da.attrs.get("units", "") or "").strip().lower()

    factor = 1.0
    if units:
        if (
            units in {"mm", "millimeter", "millimeters"} or
            "mm" in units or
            units in {"kg m-2", "kg/m^2", "kg m^-2"}
        ):
            factor = 1.0
        elif units in {"m", "meter", "meters"} or units.startswith("m "):
            factor = 1000.0
        elif units in {"in", "inch", "inches"} or "inch" in units:
            factor = 25.4

    out = da * factor
    out.attrs = dict(da.attrs)
    out.attrs["units"] = "mm"

    if verbose:
        print(f"[{source_name}] precip units='{units or 'unknown'}' -> factor={factor} -> mm")

    return out

def _build_pnnl_total_precip(ds: xr.Dataset):
    """
    Prefer a total PREC_ACC field if present.
    Otherwise sum NC + C if both exist.
    Otherwise fall back to the only PREC_ACC-like field.
    """
    vars_all = list(ds.data_vars)

    total_candidates = [
        v for v in vars_all
        if v.upper() in {"PREC_ACC", "PREC_ACC_TOT", "PREC_ACC_TOTAL"}
    ]
    if total_candidates:
        v = total_candidates[0]
        return ds[v], [v]

    nc_candidates = [v for v in vars_all if "PREC_ACC_NC" in v.upper()]
    c_candidates  = [v for v in vars_all if "PREC_ACC_C" in v.upper() and "PREC_ACC_NC" not in v.upper()]

    if nc_candidates and c_candidates:
        v_nc = nc_candidates[0]
        v_c  = c_candidates[0]
        return ds[v_nc].fillna(0) + ds[v_c].fillna(0), [v_nc, v_c]

    prec_like = [v for v in vars_all if "PREC_ACC" in v.upper()]
    if len(prec_like) == 1:
        v = prec_like[0]
        return ds[v], [v]

    if nc_candidates:
        v = nc_candidates[0]
        return ds[v], [v]

    raise RuntimeError(f"[PNNL] Could not determine total precipitation field. Vars={vars_all[:30]}")

def load_regions_gdf():
    """
    3 clipped HUC8 regions + Full Skagit boundary
    """
    import geopandas as gpd

    skagit_gdf = gpd.read_file(BOUNDARY_GEO).to_crs("EPSG:4326")
    huc8 = gpd.read_file(HUC8_GEO).to_crs("EPSG:4326")

    sub3 = ["Upper Skagit", "Sauk", "Lower Skagit"]
    huc8_3 = huc8[huc8["Name"].isin(sub3)].copy()
    if huc8_3.empty:
        raise ValueError(f"No matching HUC8 regions found for {sub3}. Check 'Name' field.")

    huc8_3 = gpd.overlay(huc8_3, skagit_gdf, how="intersection")
    regions_3 = huc8_3[["Name", "geometry"]].reset_index(drop=True)

    full_row = gpd.GeoDataFrame(
        {"Name": ["Full Skagit"], "geometry": [skagit_gdf.geometry.iloc[0]]},
        crs="EPSG:4326",
    )

    regions_gdf = pd.concat([regions_3, full_row], ignore_index=True)
    regions_gdf["Name"] = pd.Categorical(regions_gdf["Name"], categories=REGION_NAMES, ordered=True)
    regions_gdf = regions_gdf.sort_values("Name").reset_index(drop=True)
    return regions_gdf

def region_mean(da, grid_ds=None, regions_gdf=None):
    """
    Mean(da) inside each region polygon.
    overlap=True lets Full Skagit coexist with sub-basins.
    """
    if regions_gdf is None:
        raise ValueError("regions_gdf is required")

    try:
        import regionmask
    except Exception as e:
        raise RuntimeError(f"regionmask required: {e}")

    time_dim = "time" if "time" in da.dims else None
    spatial = [d for d in da.dims if d != time_dim]
    if not spatial:
        return da.expand_dims(region=list(regions_gdf["Name"].values))

    host = grid_ds if grid_ds is not None else da.to_dataset(name="_tmp")
    lat_name, lon_name = find_lat_lon_names(host)
    if not (lat_name and lon_name):
        out = da.mean(spatial, skipna=True)
        return out.expand_dims(region=list(regions_gdf["Name"].values))

    lon = _to_2d(host[lon_name])
    lat = _to_2d(host[lat_name])

    regs = regionmask.Regions(
        outlines=list(regions_gdf.geometry.values),
        names=list(regions_gdf["Name"].astype(str).values),
        numbers=list(range(len(regions_gdf))),
        name="Skagit_regions",
        overlap=True,
    )

    m3 = regs.mask_3D(lon, lat)

    spatial_dims = [d for d in da.dims if d != time_dim]
    mask_spatial_dims = [d for d in m3.dims if d != "region"]

    if set(mask_spatial_dims) != set(spatial_dims):
        rename_map = {}
        for md in mask_spatial_dims:
            for pdim in spatial_dims:
                if m3.sizes.get(md) == da.sizes.get(pdim):
                    rename_map[md] = pdim
                    break
        if rename_map:
            m3 = m3.rename(rename_map)

    out_list = []
    for i, _rname in enumerate(regs.names):
        m = m3.isel(region=i)
        w = xr.where(m, 1.0, np.nan)
        out_list.append((da * w).mean(dim=spatial_dims, skipna=True))

    out = xr.concat(out_list, dim="region").assign_coords(region=regs.names)
    return out

regions_gdf = load_regions_gdf()


def to_met_year_index(idx: pd.DatetimeIndex) -> pd.Index:
    idx = pd.DatetimeIndex(idx)
    years = idx.year.to_numpy().copy()
    years[idx.month == 12] += 1
    return pd.Index(years, name="year")

def infer_step_hours(idx: pd.DatetimeIndex):
    idx = pd.DatetimeIndex(idx).sort_values()
    if len(idx) < 2:
        return np.nan
    diffs = np.diff(idx.asi8) / 1e9 / 3600.0
    diffs = diffs[diffs > 0]
    if len(diffs) == 0:
        return np.nan
    return float(np.nanmedian(diffs))

def freq_alias_from_hours(dt_hours: float):
    if not np.isfinite(dt_hours):
        return None
    if np.isclose(dt_hours, 24.0):
        return "1D"
    if np.isclose(dt_hours, round(dt_hours)):
        return f"{int(round(dt_hours))}H"
    mins = int(round(dt_hours * 60))
    return f"{mins}min"

def regularize_series(s: pd.Series) -> pd.Series:
    s = s.sort_index()
    dt_hours = infer_step_hours(s.index)
    alias = freq_alias_from_hours(dt_hours)
    if alias is None:
        return s
    start = s.index.min().floor(alias)
    end   = s.index.max().ceil(alias)
    full_idx = pd.date_range(start, end, freq=alias)
    return s.reindex(full_idx)

def filter_year_range(series: pd.Series) -> pd.Series:
    if series.empty:
        return series
    return series[(series.index >= YEAR_MIN) & (series.index <= YEAR_MAX)]

def annual_from_daily_mm(s_mm: pd.Series):
    """
    Returns:
      annual_total_mm, annual_days_gt25_4mm
    using meteorological-year labels
    """
    s_mm = s_mm.dropna().sort_index()
    if s_mm.empty:
        return pd.Series(dtype=float), pd.Series(dtype=float)

    dt_hours = infer_step_hours(s_mm.index)
    if not np.isfinite(dt_hours):
        return pd.Series(dtype=float), pd.Series(dtype=float)

    if not np.isclose(dt_hours, 24.0):
        daily_mm = s_mm.resample("1D").sum(min_count=1)
    else:
        daily_mm = s_mm.copy()

    groups = to_met_year_index(daily_mm.index)

    counts = daily_mm.groupby(groups).count()
    annual_total_mm = daily_mm.groupby(groups).sum(min_count=1)
    annual_days_gt25_4mm = daily_mm.gt(GT_DAILY_THRESH_MM).groupby(groups).sum(min_count=1).astype(float)

    valid_years = counts[counts >= MIN_DAILY_DAYS_PER_MET_YEAR].index
    annual_total_mm = annual_total_mm.loc[annual_total_mm.index.isin(valid_years)]
    annual_days_gt25_4mm = annual_days_gt25_4mm.loc[annual_days_gt25_4mm.index.isin(valid_years)]

    annual_total_mm = filter_year_range(annual_total_mm)
    annual_days_gt25_4mm = filter_year_range(annual_days_gt25_4mm)
    return annual_total_mm, annual_days_gt25_4mm

def annual_event_max_mm(s_mm: pd.Series, duration_hours: int):
    """
    Returns annual max duration event (mm) by meteorological-year label.
    Uses rolling sum on native cadence if native cadence is <= duration.
    """
    s_mm = s_mm.dropna().sort_index()
    if s_mm.empty:
        return pd.Series(dtype=float)

    dt_hours = infer_step_hours(s_mm.index)
    if not np.isfinite(dt_hours):
        return pd.Series(dtype=float)

    if dt_hours > duration_hours + 1e-9:
        return pd.Series(dtype=float)

    ratio = duration_hours / dt_hours
    n = int(round(ratio))
    if n <= 0:
        return pd.Series(dtype=float)

    if not np.isclose(n * dt_hours, duration_hours, atol=max(1e-6, 0.05 * dt_hours)):
        return pd.Series(dtype=float)

    s_reg = regularize_series(s_mm)
    rolling_mm = s_reg.rolling(window=n, min_periods=n).sum()

    groups = to_met_year_index(rolling_mm.index)
    annual_max = rolling_mm.groupby(groups).max()

    native_groups = to_met_year_index(s_reg.index)
    counts = s_reg.groupby(native_groups).count()

    expected_steps = int(round(365.25 * 24.0 / dt_hours))
    min_steps = int(math.floor(MIN_NATIVE_COVERAGE_FRAC * expected_steps))
    valid_years = counts[counts >= min_steps].index

    annual_max = annual_max.loc[annual_max.index.isin(valid_years)]
    annual_max = filter_year_range(annual_max)
    return annual_max

def period_mean(series: pd.Series, years):
    if series.empty:
        return np.nan
    x = series[series.index.isin(years)]
    if x.empty:
        return np.nan
    return float(x.mean())

def pct_change(compare_mean, baseline_mean):
    if not np.isfinite(baseline_mean) or baseline_mean == 0 or not np.isfinite(compare_mean):
        return np.nan
    return 100.0 * (compare_mean - baseline_mean) / baseline_mean

def summarize_region_series(dataset_name, region_name, daily_s_mm, native_s_mm=None):
    """
    Returns:
      summary_row (dict),
      annual_df (DataFrame)
    """
    daily_s_mm = daily_s_mm.dropna().sort_index()
    if native_s_mm is None:
        native_s_mm = daily_s_mm.copy()
    else:
        native_s_mm = native_s_mm.dropna().sort_index()

    annual_total_mm, annual_days_gt25_4mm = annual_from_daily_mm(daily_s_mm)

    annual_max_1h  = annual_event_max_mm(native_s_mm, 1)
    annual_max_6h  = annual_event_max_mm(native_s_mm, 6)
    annual_max_24h = annual_event_max_mm(native_s_mm, 24)

    years_union = sorted(
        set(annual_total_mm.index) |
        set(annual_days_gt25_4mm.index) |
        set(annual_max_1h.index) |
        set(annual_max_6h.index) |
        set(annual_max_24h.index)
    )

    annual_rows = []
    for y in years_union:
        annual_rows.append({
            "dataset": dataset_name,
            "region": region_name,
            "year": int(y),
            "annual_total_mm": float(annual_total_mm.get(y, np.nan)),
            "annual_days_gt25_4mm": float(annual_days_gt25_4mm.get(y, np.nan)),
            "annual_max_1h_mm": float(annual_max_1h.get(y, np.nan)),
            "annual_max_6h_mm": float(annual_max_6h.get(y, np.nan)),
            "annual_max_24h_mm": float(annual_max_24h.get(y, np.nan)),
        })

    b1, c1 = period_mean(annual_max_1h, BASELINE_YEARS), period_mean(annual_max_1h, COMPARE_YEARS)
    b6, c6 = period_mean(annual_max_6h, BASELINE_YEARS), period_mean(annual_max_6h, COMPARE_YEARS)
    b24, c24 = period_mean(annual_max_24h, BASELINE_YEARS), period_mean(annual_max_24h, COMPARE_YEARS)

    summary_row = {
        "dataset": dataset_name,
        "region": region_name,
        "record_start": daily_s_mm.index.min() if not daily_s_mm.empty else pd.NaT,
        "record_end": daily_s_mm.index.max() if not daily_s_mm.empty else pd.NaT,
        "n_met_years_daily": int(len(annual_total_mm)),
        "annual_avg_total_precip_mm": float(annual_total_mm.mean()) if not annual_total_mm.empty else np.nan,
        "avg_annual_days_gt25_4mm": float(annual_days_gt25_4mm.mean()) if not annual_days_gt25_4mm.empty else np.nan,
        "baseline_mean_1h_event_mm": b1,
        "compare_mean_1h_event_mm": c1,
        "pct_change_1h": pct_change(c1, b1),
        "baseline_mean_6h_event_mm": b6,
        "compare_mean_6h_event_mm": c6,
        "pct_change_6h": pct_change(c6, b6),
        "baseline_mean_24h_event_mm": b24,
        "compare_mean_24h_event_mm": c24,
        "pct_change_24h": pct_change(c24, b24),
    }

    return summary_row, pd.DataFrame(annual_rows)

def summarize_dataset_frames(dataset_name, daily_df, native_df=None):
    summary_rows = []
    annual_parts = []

    regions_present = sorted(set(daily_df["region"].unique()))
    for region in regions_present:
        daily_s = (
            daily_df.loc[daily_df["region"] == region, ["time", "mm"]]
            .dropna()
            .sort_values("time")
            .drop_duplicates(subset=["time"], keep="first")
            .set_index("time")["mm"]
        )

        native_s = None
        if native_df is not None and not native_df.empty and region in set(native_df["region"].unique()):
            native_s = (
                native_df.loc[native_df["region"] == region, ["time", "mm"]]
                .dropna()
                .sort_values("time")
                .drop_duplicates(subset=["time"], keep="first")
                .set_index("time")["mm"]
            )

        row, ann = summarize_region_series(dataset_name, region, daily_s, native_s)
        summary_rows.append(row)
        annual_parts.append(ann)

    return pd.DataFrame(summary_rows), pd.concat(annual_parts, ignore_index=True)


def _detect_prism_var(ds):
    candidates = ["ppt", "precip", "precipitation", "pr", "tp"]
    for v in candidates:
        if v in ds.data_vars:
            return v
    for v in ds.data_vars:
        lv = v.lower()
        if ("ppt" in lv) or ("prec" in lv):
            return v
    return None

def load_prism_daily(regions_gdf):
    dataset_name = "PRISM 4km"

    prism_zarrs = []
    if P_PRISM_DIR.exists():
        prism_zarrs = sorted([p for p in P_PRISM_DIR.glob("*.zarr") if p.is_dir()])

    if not prism_zarrs:
        print(f"[{dataset_name}] no zarrs found -> skip")
        return pd.DataFrame(), pd.DataFrame()

    dsets = []
    try:
        for p in prism_zarrs:
            try:
                dsets.append(safe_open_zarr(p))
            except Exception as e:
                print(f"[{dataset_name}] failed to open {p.name}: {e}")

        if not dsets:
            return pd.DataFrame(), pd.DataFrame()

        v = None
        for ds_try in dsets:
            v = _detect_prism_var(ds_try)
            if v is not None:
                break
        if v is None:
            print(f"[{dataset_name}] could not detect precip variable")
            return pd.DataFrame(), pd.DataFrame()

        da_list = []
        for ds in dsets:
            if v not in ds:
                continue
            da = ds[v]
            if "time" not in da.dims:
                tdim = next((d for d in da.dims if ("time" in d.lower()) or ("day" in d.lower())), None)
                if tdim is not None:
                    da = da.rename({tdim: "time"})
            if "time" in da.dims:
                da_list.append(da)

        if not da_list:
            print(f"[{dataset_name}] no time-aware arrays found")
            return pd.DataFrame(), pd.DataFrame()

        prism = xr.concat(da_list, dim="time", join="outer", coords="minimal", compat="override")
        prism = ensure_time_sorted_unique(prism).sel(time=slice(GLOBAL_START, GLOBAL_END))
        prism = normalize_precip_to_mm(prism, source_name=dataset_name, verbose=True)

        rm_daily = region_mean(prism, grid_ds=prism.to_dataset(name="ppt"), regions_gdf=regions_gdf)
        daily_df = da_to_region_df(rm_daily, dataset_name)
        return daily_df, daily_df.copy()

    finally:
        for ds in dsets:
            try:
                ds.close()
            except Exception:
                pass


def ensure_daymet_pkgs():
    needed = ["pystac-client", "planetary-computer", "pyproj", "fsspec", "zarr"]
    import importlib
    import subprocess
    missing = []
    for p in needed:
        mod = p.replace("-", "_")
        try:
            importlib.import_module(mod)
        except Exception:
            missing.append(p)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

ensure_daymet_pkgs()

from pystac_client import Client
import planetary_computer as pc
from pyproj import CRS, Transformer
from shapely.ops import transform as shp_transform
from fsspec.implementations.http import HTTPFileSystem

class SASAppendingHTTPFileSystem(HTTPFileSystem):
    """
    Appends the SAS query string to every blob request correctly, so requests become:
      .../na.zarr/.zgroup?<sas>
    instead of the broken:
      .../na.zarr?<sas>/.zgroup
    """
    def __init__(self, query: str, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._fixed_query = query.lstrip("?")

    def _with_query(self, url: str) -> str:
        u = urlparse(url)
        q = u.query or self._fixed_query
        return urlunparse((u.scheme, u.netloc, u.path, u.params, q, u.fragment))

    def _open(self, path, mode="rb", block_size=None, **kwargs):
        return super()._open(self._with_query(path), mode=mode, block_size=block_size, **kwargs)

    async def _cat_file(self, url, start=None, end=None, **kwargs):
        return await super()._cat_file(self._with_query(url), start=start, end=end, **kwargs)

    async def _info(self, url, **kwargs):
        return await super()._info(self._with_query(url), **kwargs)

    async def _isfile(self, url, **kwargs):
        return await super()._isfile(self._with_query(url), **kwargs)

    async def _exists(self, url, **kwargs):
        return await super()._exists(self._with_query(url), **kwargs)

def get_daymet_mapper():
    stac = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
    col = stac.get_collection("daymet-daily-na")

    asset_key = "zarr-https" if "zarr-https" in col.assets else "zarr-abfs"
    signed = pc.sign(col.assets[asset_key]).href

    u = urlparse(signed)
    base = f"{u.scheme}://{u.netloc}{u.path}"
    query = u.query

    fs = SASAppendingHTTPFileSystem(query=query)
    return fs.get_mapper(base)

def safe_open_daymet_zarr(max_tries=DAYMET_MAX_TRIES, sleep_base=DAYMET_SLEEP_BASE):
    """
    Fresh signed mapper + retry + consolidated fallback.
    """
    last_err = None
    for attempt in range(1, max_tries + 1):
        try:
            mapper = get_daymet_mapper()
            try:
                return xr.open_zarr(mapper, consolidated=True)
            except Exception:
                mapper = get_daymet_mapper()
                return xr.open_zarr(mapper, consolidated=False)
        except Exception as e:
            last_err = e
            print(f"[Daymet-PC] open attempt {attempt}/{max_tries} failed: {type(e).__name__}: {e}")
            if attempt < max_tries:
                time.sleep(sleep_base * attempt)

    raise last_err

def daymet_lcc_crs():
    return CRS.from_proj4(
        "+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 "
        "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
    )

_DAYMET_TFM_LL_TO_XY = Transformer.from_crs("EPSG:4326", daymet_lcc_crs(), always_xy=True)

def lonlat_to_daymet_xy_bounds(geom_wgs84, pad_m=5000.0):
    minx, miny, maxx, maxy = geom_wgs84.bounds
    corners = [(minx, miny), (minx, maxy), (maxx, miny), (maxx, maxy)]
    xs, ys = zip(*[_DAYMET_TFM_LL_TO_XY.transform(lon, lat) for lon, lat in corners])
    return (
        float(min(xs) - pad_m),
        float(max(xs) + pad_m),
        float(min(ys) - pad_m),
        float(max(ys) + pad_m),
    )

def _daymet_project_geom_to_xy(geom_ll):
    def _proj(x, y, z=None):
        return _DAYMET_TFM_LL_TO_XY.transform(x, y)
    return shp_transform(_proj, geom_ll)

_DAYMET_MASK_CACHE = {}

def load_daymet_daily(regions_gdf):
    dataset_name = "Daymet-PC"

    regs = regions_gdf.to_crs("EPSG:4326")
    parts = []

    for attempt in range(1, DAYMET_MAX_TRIES + 1):
        ds = None
        try:
            ds = safe_open_daymet_zarr(max_tries=1, sleep_base=1)

            if "prcp" not in ds:
                print(f"[{dataset_name}] missing 'prcp' variable")
                return pd.DataFrame(), pd.DataFrame()

            pr_all = normalize_precip_to_mm(ds["prcp"], source_name=dataset_name, verbose=(attempt == 1))

            for rname in REGION_NAMES:
                geom_ll = regs.loc[regs["Name"] == rname, "geometry"].iloc[0]

                xmin, xmax, ymin, ymax = lonlat_to_daymet_xy_bounds(geom_ll, pad_m=5000.0)

                y0 = float(ds["y"].values[0])
                y1 = float(ds["y"].values[-1])
                y_slice = slice(ymax, ymin) if y0 > y1 else slice(ymin, ymax)

                pr = pr_all.sel(
                    time=slice(GLOBAL_START, GLOBAL_END),
                    x=slice(xmin, xmax),
                    y=y_slice,
                ).chunk({"time": 180, "y": 256, "x": 256})

                key = (
                    float(pr["x"].values[0]), float(pr["x"].values[-1]),
                    float(pr["y"].values[0]), float(pr["y"].values[-1]),
                    pr.sizes["x"], pr.sizes["y"], rname
                )

                mask = _DAYMET_MASK_CACHE.get(key)
                if mask is None:
                    xx, yy = xr.broadcast(pr["x"], pr["y"])
                    geom_xy = _daymet_project_geom_to_xy(geom_ll)

                    from shapely import contains_xy
                    mask_np = contains_xy(geom_xy, np.asarray(xx.values), np.asarray(yy.values))
                    mask = xr.DataArray(mask_np, dims=xx.dims, coords=xx.coords)
                    _DAYMET_MASK_CACHE[key] = mask

                ts = pr.where(mask).mean(dim=("y", "x"), skipna=True).compute()

                df = ts.to_dataframe(name="mm").reset_index()[["time", "mm"]]
                df["time"] = pd.to_datetime(df["time"])
                df["region"] = rname
                df["dataset"] = dataset_name
                parts.append(df[["dataset", "region", "time", "mm"]])

            if not parts:
                return pd.DataFrame(), pd.DataFrame()

            daily_df = (
                pd.concat(parts, ignore_index=True)
                  .sort_values(["region", "time"])
                  .drop_duplicates(subset=["region", "time"], keep="first")
                  .reset_index(drop=True)
            )

            return daily_df, daily_df.copy()

        except Exception as e:
            print(f"[{dataset_name}] attempt {attempt}/{DAYMET_MAX_TRIES} failed: {type(e).__name__}: {e}")
            parts = []
            if attempt < DAYMET_MAX_TRIES:
                time.sleep(DAYMET_SLEEP_BASE * attempt)
            else:
                return pd.DataFrame(), pd.DataFrame()

        finally:
            if ds is not None:
                try:
                    ds.close()
                except Exception:
                    pass


def _find_hrrr_tp_var(ds):
    if "tp" in ds.data_vars:
        return "tp"
    if "apcp" in ds.data_vars:
        return "apcp"
    for v in ds.data_vars:
        lv = v.lower()
        if lv == "tp" or "apcp" in lv or "prec" in lv:
            return v
    return None

def load_hrrr_f06(regions_gdf):
    dataset_name = "HRRR F06"

    if not P_HRRR_DIR.exists():
        print(f"[{dataset_name}] directory missing -> skip")
        return pd.DataFrame(), pd.DataFrame()

    zarrs = sorted([p for p in P_HRRR_DIR.glob("*.zarr") if p.is_dir()])
    monthly = [p for p in zarrs if re.match(r"\d{4}-\d{2}_HRRR_f06_data\.zarr", p.name)]

    if not monthly:
        print(f"[{dataset_name}] no monthly f06 zarrs found -> skip")
        return pd.DataFrame(), pd.DataFrame()

    native_pieces = []

    for p in monthly:
        ds = None
        try:
            ds = safe_open_zarr(p)
            var = _find_hrrr_tp_var(ds)
            if var is None:
                print(f"[{dataset_name}] no precip var in {p.name}")
                continue

            da = ensure_time_sorted_unique(ds[var])
            da = normalize_precip_to_mm(da, source_name=dataset_name, verbose=False)

            t = pd.to_datetime(da["time"].values)
            keep = np.isin(t.hour, [0, 6, 12, 18])
            da = da.sel(time=da["time"].values[keep])

            da = da.assign_coords(time=(pd.to_datetime(da["time"].values) + pd.Timedelta(hours=6)))
            da = da.sel(time=slice(GLOBAL_START, GLOBAL_END))

            if da.sizes.get("time", 0) == 0:
                continue

            rm_native = region_mean(da, grid_ds=ds, regions_gdf=regions_gdf)
            native_pieces.append(rm_native)

        except Exception as e:
            print(f"[{dataset_name}] {p.name} failed: {type(e).__name__}: {e}")
        finally:
            if ds is not None:
                try:
                    ds.close()
                except Exception:
                    pass

    if not native_pieces:
        return pd.DataFrame(), pd.DataFrame()

    native = xr.concat(native_pieces, dim="time", join="outer", coords="minimal", compat="override")
    native = ensure_time_sorted_unique(native).sel(time=slice(GLOBAL_START, GLOBAL_END))
    daily = native.resample(time="1D").sum()

    native_df = da_to_region_df(native, dataset_name)
    daily_df  = da_to_region_df(daily, dataset_name)
    return daily_df, native_df


def pick_time_dim(da):
    if "time" in da.dims:
        return "time"
    if "day" in da.dims:
        return "day"
    for d in da.dims:
        if "time" in d.lower() or "day" in d.lower():
            return d
    raise ValueError(f"[UCLA] No time-like dim found. dims={da.dims}")

def ensure_sorted_unique_time(da, tdim="time"):
    da = da.sortby(tdim)
    try:
        t = pd.to_datetime(da[tdim].values)
        _, idx = np.unique(t, return_index=True)
        da = da.isel({tdim: np.sort(idx)})
    except Exception:
        pass
    return da

def build_ucla_region_masks(coord_file: Path, regions_gdf):
    import shapely
    try:
        from shapely import contains_xy
        has_contains_xy = True
    except Exception:
        has_contains_xy = False
        from shapely.prepared import prep

    g = xr.open_dataset(coord_file, engine="netcdf4")
    lat = g["lat2d"].squeeze(drop=True).astype("float64")
    lon = g["lon2d"].squeeze(drop=True).astype("float64")
    g.close()

    lonv = lon.values.ravel()
    latv = lat.values.ravel()
    ny, nx = lon.shape

    masks = {}
    for _, row in regions_gdf.iterrows():
        name = str(row["Name"])
        geom = row["geometry"]

        if has_contains_xy:
            m_flat = contains_xy(geom, lonv, latv)
        else:
            pg = prep(geom)
            m_flat = np.array([pg.contains(shapely.Point(x, y)) for x, y in zip(lonv, latv)], dtype=bool)

        mask2d = m_flat.reshape(ny, nx).astype("float32")
        masks[name] = xr.DataArray(mask2d, dims=lon.dims, coords=lon.coords)

    return masks

def masked_mean_timeseries(pr: xr.DataArray, mask2d: xr.DataArray) -> xr.DataArray:
    spatial_dims = [d for d in pr.dims if d != "time"]

    mask = mask2d
    if set(mask.dims) != set(spatial_dims):
        rename_map = {}
        for md in mask.dims:
            for pdim in spatial_dims:
                if mask.sizes[md] == pr.sizes[pdim]:
                    rename_map[md] = pdim
                    break
        mask = mask.rename(rename_map)

    denom = mask.sum(dim=spatial_dims, skipna=True)
    if float(denom.values) == 0.0:
        return xr.full_like(pr.isel(time=0), np.nan).expand_dims({"time": pr["time"]})

    pr = pr.chunk({"time": min(366, pr.sizes["time"])})
    num = (pr * mask).sum(dim=spatial_dims, skipna=True)
    return num / denom

def load_ucla_daily(regions_gdf):
    dataset_name = "UCLA ERA5 d02"

    if not UCLA_PREC_DIR.exists() or not UCLA_COORD_FILE.exists():
        print(f"[{dataset_name}] missing precip dir or coord file -> skip")
        return pd.DataFrame(), pd.DataFrame()

    masks = build_ucla_region_masks(UCLA_COORD_FILE, regions_gdf)
    files = sorted(glob(str(UCLA_PREC_DIR / "prec.daily.era5.d02.*.nc")))
    if not files:
        print(f"[{dataset_name}] no files found -> skip")
        return pd.DataFrame(), pd.DataFrame()

    parts = []
    for f in files:
        ds = None
        try:
            ds = xr.open_dataset(f, engine="netcdf4", decode_cf=True, mask_and_scale=True)
            if "prec" not in ds:
                continue

            pr = ds["prec"]
            tdim = pick_time_dim(pr)
            if tdim != "time":
                pr = pr.rename({tdim: "time"})

            pr = ensure_sorted_unique_time(pr, "time").sel(time=slice(GLOBAL_START, GLOBAL_END))
            pr = normalize_precip_to_mm(pr, source_name=dataset_name, verbose=False)

            if pr.sizes.get("time", 0) == 0:
                continue

            for rname in REGION_NAMES:
                ts = masked_mean_timeseries(pr, masks[rname]).compute()
                df = ts.to_dataframe(name="mm").reset_index()[["time", "mm"]]
                df["time"] = pd.to_datetime(df["time"])
                df["region"] = rname
                df["dataset"] = dataset_name
                parts.append(df[["dataset", "region", "time", "mm"]])

        except Exception as e:
            print(f"[{dataset_name}] {Path(f).name} failed: {type(e).__name__}: {e}")
        finally:
            if ds is not None:
                try:
                    ds.close()
                except Exception:
                    pass

    if not parts:
        return pd.DataFrame(), pd.DataFrame()

    daily_df = (
        pd.concat(parts, ignore_index=True)
        .sort_values(["region", "time"])
        .drop_duplicates(subset=["region", "time"], keep="first")
        .reset_index(drop=True)
    )
    return daily_df, daily_df.copy()


def load_snotel_daily(regions_gdf):
    dataset_name = "SNOTEL (mean stations)"

    try:
        import geopandas as gpd
        import shapely.geometry as sgeom
        import requests
    except Exception as e:
        print(f"[{dataset_name}] missing dependency: {e}")
        return pd.DataFrame(), pd.DataFrame()

    stations_url = "https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/all_stations.geojson"
    try:
        r = requests.get(stations_url, timeout=30)
        r.raise_for_status()
        gj = r.json()
    except Exception as e:
        print(f"[{dataset_name}] failed to download stations metadata: {e}")
        return pd.DataFrame(), pd.DataFrame()

    records, geoms = [], []
    for feat in gj.get("features", []):
        props = feat.get("properties", {})
        geom = feat.get("geometry")
        if geom is None:
            continue
        records.append(props)
        geoms.append(sgeom.shape(geom))

    if not records:
        return pd.DataFrame(), pd.DataFrame()

    stations = gpd.GeoDataFrame(records, geometry=geoms, crs="EPSG:4326")
    skagit_geom = gpd.read_file(BOUNDARY_GEO).to_crs("EPSG:4326").geometry.iloc[0]
    inside = stations.geometry.within(skagit_geom)
    skagit_stations = stations[inside].copy()

    desired_names = {"Beaver Pass", "Brown Top", "Marten Ridge", "Rainy Pass", "Swamp Creek", "Thunder Basin"}
    if "name" in skagit_stations.columns:
        skagit_stations = skagit_stations[skagit_stations["name"].isin(desired_names)]

    if skagit_stations.empty:
        print(f"[{dataset_name}] no stations found after filtering")
        return pd.DataFrame(), pd.DataFrame()

    regs3 = regions_gdf[regions_gdf["Name"].isin(["Upper Skagit", "Sauk", "Lower Skagit"])].copy().to_crs("EPSG:4326")
    try:
        joined = gpd.sjoin(skagit_stations, regs3.rename(columns={"Name": "region"}), predicate="within", how="left")
    except TypeError:
        joined = gpd.sjoin(skagit_stations, regs3.rename(columns={"Name": "region"}), op="within", how="left")

    station_region_map = {}
    for _, row in joined.dropna(subset=["region"]).iterrows():
        code = row.get("code")
        region = row.get("region")
        if pd.notna(code):
            station_region_map[str(code)] = str(region)

    base_csv_url = "https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/data"
    station_series = {}

    for _, st in skagit_stations.iterrows():
        code = st.get("code")
        if pd.isna(code):
            continue
        code = str(code)
        csv_url = f"{base_csv_url}/{code}.csv"
        try:
            df = pd.read_csv(csv_url, index_col="datetime", parse_dates=True)
        except Exception as e:
            print(f"[{dataset_name}] failed to read {code}: {e}")
            continue

        if "PRCPSA" not in df.columns:
            continue

        # keep the same notebook logic; values are treated as mm here
        s = (df["PRCPSA"] * 1000.0).sort_index()
        s = s.loc[GLOBAL_START:GLOBAL_END]

        dt = infer_step_hours(pd.DatetimeIndex(s.index))
        if np.isfinite(dt) and dt < 24:
            s = s.resample("1D").sum(min_count=1)

        station_series[code] = s

    if not station_series:
        return pd.DataFrame(), pd.DataFrame()

    def mean_series(codes):
        use = [station_series[c] for c in codes if c in station_series]
        if not use:
            return pd.Series(dtype=float)
        mat = pd.concat(use, axis=1)
        return mat.mean(axis=1, skipna=True).sort_index()

    region_codes = {
        region: [code for code, reg in station_region_map.items() if reg == region]
        for region in ["Upper Skagit", "Sauk", "Lower Skagit"]
    }
    full_codes = list(station_series.keys())

    parts = []
    for region in ["Upper Skagit", "Sauk", "Lower Skagit"]:
        s = mean_series(region_codes.get(region, []))
        if s.empty:
            continue
        parts.append(series_to_time_df(s, dataset_name, region))

    s_full = mean_series(full_codes)
    if not s_full.empty:
        parts.append(series_to_time_df(s_full, dataset_name, "Full Skagit"))

    if not parts:
        return pd.DataFrame(), pd.DataFrame()

    daily_df = (
        pd.concat(parts, ignore_index=True)
          .sort_values(["region", "time"])
          .drop_duplicates(subset=["region", "time"], keep="first")
          .reset_index(drop=True)
    )
    return daily_df, daily_df.copy()


def _find_pnnl_prec_acc_nc_var(ds):
    """
    Match the seasonal notebook behavior:
    prefer PREC_ACC_NC only.
    """
    exact = [v for v in ds.data_vars if v.upper() == "PREC_ACC_NC"]
    if exact:
        return exact[0]

    cands = [v for v in ds.data_vars if "PREC_ACC_NC" in v.upper()]
    if cands:
        return cands[0]

    return None


def pnnl_region_mean_with_mask(p, mask_ds):
    """
    Fallback mask-based regional averaging, matching the seasonal notebook.
    Usually yields only the regions present in the mask file.
    """
    if "mask" not in mask_ds:
        raise ValueError("[PNNL] Expected variable 'mask' in fallback region mask")

    region_labels = [str(r) for r in mask_ds["region"].values]
    spatial_dims = [d for d in p.dims if d != "time"]

    out_list = []
    out_regions = []

    for rname in region_labels:
        mr = mask_ds["mask"].sel(region=rname)

        if set(mr.dims) != set(spatial_dims):
            rename_map = {}
            for md in mr.dims:
                for pdim in spatial_dims:
                    if mr.sizes[md] == p.sizes[pdim]:
                        rename_map[md] = pdim
                        break
            if rename_map:
                mr = mr.rename(rename_map)

        ts = p.where(mr).mean(dim=spatial_dims, skipna=True)
        out_list.append(ts)
        out_regions.append(rname)

    out = xr.concat(out_list, dim="region").assign_coords(region=out_regions)
    return out


def load_pnnl_native(regions_gdf):
    """
    PNNL handling aligned to your seasonal notebook:
      - read only *PREC_ACC_NC*.nc
      - keep only PREC_ACC_NC-like vars
      - prefer fallback regional mask when present
      - do NOT reconstruct total precip using other PREC_ACC vars
    """
    dataset_name = "PNNL hist"

    if not PNNL_HIST.exists():
        print(f"[{dataset_name}] directory missing -> skip")
        return pd.DataFrame(), pd.DataFrame()

    mask_ds = None
    use_mask_fallback = False

    if PNNL_REGION_MASK.exists():
        try:
            mask_ds = xr.open_dataset(PNNL_REGION_MASK)
            if "mask" in mask_ds and "region" in mask_ds["mask"].dims:
                use_mask_fallback = True
                print(f"[{dataset_name}] using fallback mask file: {PNNL_REGION_MASK}")
            else:
                print(f"[{dataset_name}] mask file found but invalid structure -> using region_mean fallback")
        except Exception as e:
            print(f"[{dataset_name}] failed to open fallback mask: {e}")
            mask_ds = None

    native_pieces = []
    units_logged = False

    year_dirs = sorted([p for p in PNNL_HIST.glob("*") if p.is_dir() and p.name.isdigit()])
    if not year_dirs:
        print(f"[{dataset_name}] no yearly directories found")
        return pd.DataFrame(), pd.DataFrame()

    for yd in year_dirs:
        yy = int(yd.name)

        # Match the seasonal notebook exactly: use only PREC_ACC_NC files
        files = sorted(glob(str(yd / "*PREC_ACC_NC*.nc")))
        if not files:
            continue

        def _preprocess(ds):
            keep = [v for v in ds.data_vars if "PREC_ACC_NC" in v.upper()]
            return ds[keep] if keep else ds

        ds = None
        try:
            ds = xr.open_mfdataset(
                files,
                combine="by_coords",
                engine="netcdf4",
                parallel=False,
                preprocess=_preprocess,
                chunks={"time": 24 * 7},
            )

            v = _find_pnnl_prec_acc_nc_var(ds)
            if v is None:
                print(f"[{dataset_name}] year {yy}: no PREC_ACC_NC variable found")
                continue

            p = ensure_time_sorted_unique(ds[v]).sel(time=slice(GLOBAL_START, GLOBAL_END))
            if p.sizes.get("time", 0) == 0:
                continue

            p = normalize_precip_to_mm(
                p,
                source_name=f"{dataset_name} var={v}",
                verbose=(not units_logged),
            )
            units_logged = True

            # Prefer the same mask-based route as the seasonal notebook
            if use_mask_fallback:
                rm_native = pnnl_region_mean_with_mask(p, mask_ds)
            else:
                rm_native = region_mean(p, grid_ds=ds, regions_gdf=regions_gdf)

            native_pieces.append(rm_native)

        except Exception as e:
            print(f"[{dataset_name}] year {yy} failed: {type(e).__name__}: {e}")
        finally:
            if ds is not None:
                try:
                    ds.close()
                except Exception:
                    pass

    if mask_ds is not None:
        try:
            mask_ds.close()
        except Exception:
            pass

    if not native_pieces:
        return pd.DataFrame(), pd.DataFrame()

    native = xr.concat(native_pieces, dim="time", join="outer", coords="minimal", compat="override")
    native = ensure_time_sorted_unique(native).sel(time=slice(GLOBAL_START, GLOBAL_END))

    if "region" in native.coords:
        present_regions = [r for r in REGION_NAMES if r in set(native["region"].values)]
        if present_regions:
            native = native.sel(region=present_regions)

    # Native PNNL is sub-daily; convert to daily totals
    daily = native.resample(time="1D").sum(min_count=1)

    native_df = da_to_region_df(native, dataset_name)
    daily_df = da_to_region_df(daily, dataset_name)

    return daily_df, native_df


def load_conus404_daily(regions_gdf):
    dataset_name = "CONUS404"

    try:
        import intake
        from shapely import contains_xy
    except Exception as e:
        print(f"[{dataset_name}] missing dependency: {e}")
        return pd.DataFrame(), pd.DataFrame()

    url = "https://raw.githubusercontent.com/hytest-org/hytest/main/dataset_catalog/hytest_intake_catalog.yml"

    try:
        cat = intake.open_catalog(url)
        ds = cat["conus404-catalog"]["conus404-daily-osn"].to_dask()
    except Exception as e:
        print(f"[{dataset_name}] failed to open intake catalog: {e}")
        return pd.DataFrame(), pd.DataFrame()

    if "PREC_ACC_NC" not in ds.data_vars:
        print(f"[{dataset_name}] missing PREC_ACC_NC")
        return pd.DataFrame(), pd.DataFrame()

    precip = ds["PREC_ACC_NC"].sel(time=slice(GLOBAL_START, GLOBAL_END))
    precip = normalize_precip_to_mm(precip, source_name=dataset_name, verbose=True)

    lat = ds["lat"]
    lon = ds["lon"]

    parts = []
    for region in REGION_NAMES:
        try:
            geom = regions_gdf.loc[regions_gdf["Name"] == region, "geometry"].iloc[0]
            mask_np = contains_xy(geom, lon.values, lat.values)
            mask_da = xr.DataArray(mask_np, dims=lat.dims, coords=lat.coords)

            ts = precip.where(mask_da).mean(dim=("y", "x"), skipna=True).compute()
            df = ts.to_dataframe(name="mm").reset_index()[["time", "mm"]]
            df["time"] = pd.to_datetime(df["time"])
            df["region"] = region
            df["dataset"] = dataset_name
            parts.append(df[["dataset", "region", "time", "mm"]])
        except Exception as e:
            print(f"[{dataset_name}] region {region} failed: {type(e).__name__}: {e}")

    if not parts:
        return pd.DataFrame(), pd.DataFrame()

    daily_df = pd.concat(parts, ignore_index=True).sort_values(["region", "time"]).reset_index(drop=True)
    return daily_df, daily_df.copy()


all_summary = []
all_annual = []

def run_one(loader_name, loader_func):
    print(f"\n{'='*70}\nRunning {loader_name}\n{'='*70}")
    t0 = time.time()
    daily_df, native_df = loader_func(regions_gdf)
    if daily_df is None or daily_df.empty:
        print(f"[{loader_name}] no data returned")
        return
    summary_df, annual_df = summarize_dataset_frames(loader_name, daily_df, native_df)
    all_summary.append(summary_df)
    all_annual.append(annual_df)
    print(f"[{loader_name}] done in {(time.time() - t0)/60.0:.2f} min")

if RUN_DATASETS["PRISM_4KM"]:
    run_one("PRISM 4km", load_prism_daily)

if RUN_DATASETS["DAYMET_PC"]:
    run_one("Daymet-PC", load_daymet_daily)

if RUN_DATASETS["HRRR_F06"]:
    run_one("HRRR F06", load_hrrr_f06)

if RUN_DATASETS["UCLA_ERA5_D02"]:
    run_one("UCLA ERA5 d02", load_ucla_daily)

if RUN_DATASETS["PNNL_HIST"]:
    run_one("PNNL hist", load_pnnl_native)

if RUN_DATASETS["CONUS404"]:
    run_one("CONUS404", load_conus404_daily)

if RUN_DATASETS["SNOTEL"]:
    run_one("SNOTEL (mean stations)", load_snotel_daily)

if not all_summary:
    raise RuntimeError("No dataset produced output.")

summary_df = pd.concat(all_summary, ignore_index=True)
annual_df  = pd.concat(all_annual, ignore_index=True)


annual_df = annual_df.sort_values(["region", "dataset", "year"]).reset_index(drop=True)
summary_df = summary_df.sort_values(["region", "dataset"]).reset_index(drop=True)

annual_df.to_csv(ANNUAL_CSV, index=False)
summary_df.to_csv(SUMMARY_CSV, index=False)

print("Saved annual metrics ->", ANNUAL_CSV)
print("Saved summary metrics ->", SUMMARY_CSV)


metric_name_map = {
    "annual_avg_total_precip_mm": "Annual Average Total Precipitation (mm)",
    "avg_annual_days_gt25_4mm": f"Average Annual Number of Days with >{GT_DAILY_THRESH_MM:.1f} mm Total Precipitation",
    "pct_change_1h": "Average Percent Change in 1 Hour Duration Precipitation Event",
    "pct_change_6h": "Average Percent Change in 6 Hour Duration Precipitation Event",
    "pct_change_24h": "Average Percent Change in 24 Hour Duration Precipitation Event",
}

metric_order = [
    "annual_avg_total_precip_mm",
    "avg_annual_days_gt25_4mm",
    "pct_change_1h",
    "pct_change_6h",
    "pct_change_24h",
]

for region in REGION_NAMES:
    sub = summary_df[summary_df["region"] == region].copy()
    if sub.empty:
        continue

    wide = sub.set_index("dataset")[metric_order].T
    wide.index = [metric_name_map[x] for x in wide.index]

    print("\n" + "="*100)
    print(f"REGION: {region}")
    print("="*100)
    display(wide.round(2))


event_debug_cols = [
    "dataset", "region",
    "baseline_mean_1h_event_mm", "compare_mean_1h_event_mm", "pct_change_1h",
    "baseline_mean_6h_event_mm", "compare_mean_6h_event_mm", "pct_change_6h",
    "baseline_mean_24h_event_mm", "compare_mean_24h_event_mm", "pct_change_24h",
]
display(summary_df[event_debug_cols].round(2))


for region in REGION_NAMES:
    sub = summary_df[summary_df["region"] == region].copy()
    if sub.empty:
        continue

    wide = sub.set_index("dataset")[metric_order].T
    wide.index = [metric_name_map[x] for x in wide.index]

    out_csv = OUT / f"summary_table_{region.replace(' ', '_')}_{YEAR_MIN}_{YEAR_MAX}.csv"
    wide.to_csv(out_csv)
    print("Saved region summary table ->", out_csv)


Running PRISM 4km
[PRISM 4km] precip units='unknown' -> factor=1.0 -> mm
[PRISM 4km] done in 0.08 min

Running Daymet-PC
[Daymet-PC] precip units='mm/day' -> factor=1.0 -> mm
[Daymet-PC] done in 20.08 min

Running HRRR F06


Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7b6349b2eab0>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x7b61e920e4b0>, 1590597.934512084)])']
connector: <aiohttp.connector.TCPConnector object at 0x7b61ee87ce30>
Future exception was never retrieved
future: <Future finished exception=ClientConnectionError('Connection lost: SSL shutdown timed out')>
TimeoutError: SSL shutdown timed out

The above exception was the direct cause of the following exception:

aiohttp.client_exceptions.ClientConnectionError: Connection lost: SSL shutdown timed out


[HRRR F06] done in 2.31 min

Running UCLA ERA5 d02
[UCLA ERA5 d02] done in 1.53 min

Running PNNL hist
[PNNL hist] using fallback mask file: /home/balaji24/skagit-met/analysis/skagit_huc8_3mask_pnnl_450x450.nc
[PNNL hist var=PREC_ACC_NC] precip units='mm' -> factor=1.0 -> mm
[PNNL hist] year 1982 failed: ValueError: broadcasting cannot handle duplicate dimensions on a variable: ['x', 'x']
[PNNL hist] year 1983 failed: ValueError: broadcasting cannot handle duplicate dimensions on a variable: ['x', 'x']


/home/balaji24/skagit-met/conus-env/lib/python3.12/site-packages/xarray/namedarray/core.py:500: UserWarning: Duplicate dimension names present: dimensions {'x'} appear more than once in dims=('x', 'x'). We do not yet support duplicate dimension names, but we do allow initial construction of the object. We recommend you rename the dims immediately to become distinct, as most xarray functionality is likely to fail silently if you do not. To rename the dimensions you will need to set the ``.dims`` attribute of each variable, ``e.g. var.dims=('x0', 'x1')``.
  self._dims = self._parse_dimensions(value)
/home/balaji24/skagit-met/conus-env/lib/python3.12/site-packages/xarray/namedarray/core.py:261: UserWarning: Duplicate dimension names present: dimensions {'x'} appear more than once in dims=('x', 'x'). We do not yet support duplicate dimension names, but we do allow initial construction of the object. We recommend you rename the dims immediately to become distinct, as most xarray functionali

[PNNL hist] year 1984 failed: ValueError: broadcasting cannot handle duplicate dimensions on a variable: ['x', 'x']
[PNNL hist] year 1985 failed: ValueError: broadcasting cannot handle duplicate dimensions on a variable: ['x', 'x']


...

dataset,Daymet-PC,HRRR F06,PRISM 4km,UCLA ERA5 d02
Annual Average Total Precipitation (mm),2204.19,1687.95,2340.65,1861.92
Average Annual Number of Days with >25.4 mm Total Precipitation,14.05,15.60,22.93,16.45
Average Percent Change in 1 Hour Duration Precipitation Event,NaN,NaN,NaN,NaN
Average Percent Change in 6 Hour Duration Precipitation Event,NaN,NaN,NaN,NaN
Average Percent Change in 24 Hour Duration Precipitation Event,NaN,NaN,-15.49,-3.90



REGION: Full Skagit


dataset,Daymet-PC,HRRR F06,PRISM 4km,SNOTEL (mean stations),UCLA ERA5 d02
Annual Average Total Precipitation (mm),2111.76,1758.94,2360.01,2055.40,2440.70
Average Annual Number of Days with >25.4 mm Total Precipitation,15.97,17.00,23.00,17.69,26.12
Average Percent Change in 1 Hour Duration Precipitation Event,NaN,NaN,NaN,NaN,NaN
Average Percent Change in 6 Hour Duration Precipitation Event,NaN,NaN,NaN,NaN,NaN
Average Percent Change in 24 Hour Duration Precipitation Event,NaN,NaN,-11.04,14.47,-4.52


,dataset,region,baseline_mean_1h_event_mm,compare_mean_1h_event_mm,pct_change_1h,baseline_mean_6h_event_mm,compare_mean_6h_event_mm,pct_change_6h,baseline_mean_24h_event_mm,compare_mean_24h_event_mm,pct_change_24h
0,Daymet-PC,Full Skagit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HRRR F06,Full Skagit,NaN,NaN,NaN,NaN,27.13,NaN,NaN,76.52,NaN
2,PRISM 4km,Full Skagit,NaN,NaN,NaN,NaN,NaN,NaN,85.30,75.88,-11.04
3,SNOTEL (mean stations),Full Skagit,NaN,NaN,NaN,NaN,NaN,NaN,77.32,88.51,14.47
4,UCLA ERA5 d02,Full Skagit,NaN,NaN,NaN,NaN,NaN,NaN,79.33,75.74,-4.52
5,Daymet-PC,Lower Skagit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,HRRR F06,Lower Skagit,NaN,NaN,NaN,NaN,27.10,NaN,NaN,70.33,NaN
7,PRISM 4km,Lower Skagit,NaN,NaN,NaN,NaN,NaN,NaN,82.45,69.68,-15.49
8,UCLA ERA5 d02,Lower Skagit,NaN,NaN,NaN,NaN,NaN,NaN,67.28,64.66,-3.90
9,Daymet-PC,Sauk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved region summary table -> /data0/balaji24/data/derived/precip_summary_metrics/summary_table_Upper_Skagit_1983_2024.csv
Saved region summary table -> /data0/balaji24/data/derived/precip_summary_metrics/summary_table_Sauk_1983_2024.csv
Saved region summary table -> /data0/balaji24/data/derived/precip_summary_metrics/summary_table_Lower_Skagit_1983_2024.csv
Saved region summary table -> /data0/balaji24/data/derived/precip_summary_metrics/summary_table_Full_Skagit_1983_2024.csv
